# PSO para reglas anomalas basadas en similaridad

Notebook para descubrir reglas de excepcion condicionadas respecto a reglas dominantes.


# PSO para reglas anomalas basadas en similaridad

## 1. Proposito y flujo metodologico

Este notebook busca reglas anomalas entendidas como excepciones condicionadas dentro del contexto de una regla dominante.

La regla dominante se denota por:

$$
R_d: X \Rightarrow Y
$$

donde $X$ es el antecedente, $Y$ es el consecuente binario y $R_d$ es la regla de referencia.

Para cada regla dominante se mantiene fijo $X$ y se busca un conjunto adicional de condiciones $Z$:

$$
R_a: X \land Z \Rightarrow \neg Y
$$

En esta notacion:

- $Z$ contiene entre una y tres condiciones adicionales.
- $\operatorname{Variables}(X) \cap \operatorname{Variables}(Z)=\varnothing$.
- Las condiciones continuas usan un centro $c_j$ y un radio $r_j$.
- Las condiciones categoricas usan igualdad exacta.
- $A_X(i)$, $A_Z(i)$ y $A_{XZ}(i)$ representan las activaciones de $X$, $Z$ y su combinacion.
- `pBest` y `gBest` se refieren a la mejor particula individual y global del PSO anomaloso.

La activacion condicionada se calcula mediante:

$$
A_{XZ}(i)=\min\left(A_X(i),A_Z(i)\right)
$$

El soporte anomaloso y las dos confianzas condicionadas son:

$$
Supp_a=Supp(X\land Z\land \neg Y)
$$

$$
Conf_X(Z\Rightarrow\neg Y)=
\frac{Supp(X\land Z\land \neg Y)}{Supp(X\land Z)}
$$

$$
Conf_X(\neg Y\Rightarrow Z)=
\frac{Supp(X\land Z\land \neg Y)}{Supp(X\land \neg Y)}
$$

Cada confianza se compara con su frecuencia base mediante el factor de certeza condicionado $CF_X$. La funcion de evaluacion de una candidata se denota por $F_a$ y combina evidencia, desviacion respecto a la regla dominante y parsimonia.

El flujo del notebook es:

1. Cargar el dataset normalizado $D'$, el dataset original y el catalogo de reglas dominantes.
2. Reconstruir $R_d$ y verificar sus metricas.
3. Optimizar $Z$ con PSO para formar $R_a$.
4. Consolidar candidatas mediante Jaccard difuso.
5. Validar estabilidad con nuevas semillas.
6. Generar catalogos y resultados finales.

Las pruebas individuales, la comparacion exploratoria y las graficas historicas permanecen disponibles mediante banderas independientes, pero no alteran el flujo principal cuando estan desactivadas.


## 2. Carga de datos y catalogo dominante

Se cargan el dataset normalizado $D'$, el dataset original y los metadatos de normalizacion. El catalogo principal contiene las reglas dominantes $R_d$ que seran usadas como contexto para buscar reglas anomalas $R_a$.

La representacion normalizada se utiliza para evaluar similaridad; el dataset original queda disponible para interpretar las condiciones en sus unidades de origen.


In [ ]:
import copy
import gc
import os
from itertools import product

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ============================================================
# CARGAR DATOS Y REGLAS DOMINANTES
# ============================================================

# 1. Localizar la raiz del repositorio

from pathlib import Path


def localizar_raiz_repositorio():
    """Encuentra la carpeta que contiene notebooks/ del repositorio."""
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / "notebooks").is_dir():
            return candidato
    return Path.cwd()


raiz_repositorio = localizar_raiz_repositorio()
carpeta_artifacts = raiz_repositorio / "artifacts"
carpeta_base = carpeta_artifacts / "anomalo"
carpeta_preprocesamiento = carpeta_artifacts / "preprocesamiento"
carpeta_dominante = (
    carpeta_artifacts / "dominante" / "experimento_pso_reglas_dominantes"
)

rutas = {
    "normalizado": carpeta_preprocesamiento / "cardiovascular_normalizado_robusto.csv",
    "original": carpeta_preprocesamiento / "cardiovascular_preparado_original.csv",
    "metadatos": carpeta_preprocesamiento / "metadatos_normalizacion_robusta.pkl",
    "dominantes": carpeta_dominante / "resultados_reglas_dominantes.pkl"
}

# 3. Verificar archivos
archivos_faltantes = [ruta for ruta in rutas.values() if not os.path.exists(ruta)]

if archivos_faltantes:
    raise FileNotFoundError(f"No se encontraron estos archivos: {archivos_faltantes}")

# 4. Cargar datos y resultados dominantes

df_similitud = pd.read_csv(rutas["normalizado"])
df_original = pd.read_csv(rutas["original"])
metadatos_normalizacion = joblib.load(rutas["metadatos"])
resultados_dominantes = joblib.load(rutas["dominantes"])

# 5. Recuperar metadatos
parametros_normalizacion_robusta = metadatos_normalizacion["parametros_normalizacion"]
mapas_categorias = metadatos_normalizacion["mapas_categorias"]
columnas_continuas = metadatos_normalizacion["columnas_continuas"]
columnas_categoricas = metadatos_normalizacion["columnas_categoricas"]
columna_objetivo = metadatos_normalizacion["columna_objetivo"]
columnas_antecedente = columnas_continuas + columnas_categoricas

# 6. Seleccionar catalogos a partir del resultado dominante

catalogo_dominantes_reproducible = (
    resultados_dominantes["catalogo_reproducible"].copy()
)
reglas_dominantes_reproducibles = (
    resultados_dominantes["reglas_dominantes_reproducibles"]
)

catalogo_dominantes_principal = (
    catalogo_dominantes_reproducible[
        catalogo_dominantes_reproducible["nivel_estabilidad"] == "alta"
    ]
    .copy()
    .reset_index(drop=True)
)

reglas_por_id = {
    regla["regla_id"]: regla
    for regla in reglas_dominantes_reproducibles
}

reglas_dominantes_principales = []
for regla_id in catalogo_dominantes_principal["regla_id"]:
    if regla_id not in reglas_por_id:
        raise KeyError(
            f"No existe la estructura de la regla dominante {regla_id}."
        )
    reglas_dominantes_principales.append(reglas_por_id[regla_id])

catalogo_dominantes_ampliado = catalogo_dominantes_reproducible.copy()
reglas_dominantes_ampliadas = list(reglas_dominantes_reproducibles)

# 7. Verificar correspondencia
if len(df_similitud) != len(df_original):
    raise ValueError("Los datasets normalizado y original tienen diferente número de registros.")

if len(catalogo_dominantes_principal) != len(reglas_dominantes_principales):
    raise ValueError("El catálogo principal y sus estructuras no tienen la misma longitud.")

if len(catalogo_dominantes_ampliado) != len(reglas_dominantes_ampliadas):
    raise ValueError("El catálogo ampliado y sus estructuras no tienen la misma longitud.")

# 8. Mostrar confirmación
print(f"Dataset normalizado: {df_similitud.shape}")
print(f"Dataset original: {df_original.shape}")
print(f"Reglas dominantes principales: {len(reglas_dominantes_principales)}")
print(f"Reglas dominantes ampliadas: {len(reglas_dominantes_ampliadas)}")
print(f"Variable objetivo: {columna_objetivo}")

In [ ]:
# ============================================================
# DEFINIR FUNCIONES DE SIMILITUD
# ============================================================

def similitud_triangular(x, centro, radio):
    """Calcula la similaridad triangular."""
    if radio <= 0:
        raise ValueError("El radio debe ser mayor que cero.")

    x = np.asarray(x, dtype=float)
    return np.maximum(0.0, 1.0 - np.abs(x - centro) / radio)


def similitud_gaussiana_truncada(x, centro, radio):
    """Calcula la similaridad gaussiana truncada en tres sigmas."""
    if radio <= 0:
        raise ValueError("El radio debe ser mayor que cero.")

    x = np.asarray(x, dtype=float)
    distancia = np.abs(x - centro)
    sigma = radio / 3.0
    similitud = np.exp(-0.5 * (distancia / sigma) ** 2)

    return np.where(distancia < radio, similitud, 0.0)


def similitud_categorica(x, valor):
    """Evalúa una condición categórica mediante igualdad exacta."""
    return (np.asarray(x) == valor).astype(float)

In [ ]:
# ============================================================
# CALCULAR ACTIVACIÓN DE CONDICIONES
# ============================================================

def calcular_activacion_condicion(dataframe, condicion, metodo_similitud):
    """Calcula la activación de una condición continua o categórica."""
    variable = condicion["variable"]

    if variable not in dataframe.columns:
        raise ValueError(f"La variable '{variable}' no existe en el dataset.")

    valores = dataframe[variable].to_numpy()

    if condicion["tipo"] == "continua":
        centro = float(condicion["centro"])
        radio = float(condicion["radio"])

        if metodo_similitud == "triangular":
            return similitud_triangular(valores, centro, radio)

        if metodo_similitud == "gaussiana":
            return similitud_gaussiana_truncada(valores, centro, radio)

        raise ValueError(f"Método de similaridad desconocido: {metodo_similitud}")

    if condicion["tipo"] == "categorica":
        return similitud_categorica(valores, condicion["valor"])

    raise ValueError(f"Tipo de condición desconocido: {condicion['tipo']}")


def calcular_activacion_antecedente(
    dataframe,
    condiciones,
    metodo_similitud,
    devolver_detalle=False
):
    """Combina las condiciones de un antecedente mediante el mínimo."""
    if not condiciones:
        raise ValueError("El antecedente debe contener al menos una condición.")

    variables = [condicion["variable"] for condicion in condiciones]

    if len(variables) != len(set(variables)):
        raise ValueError("Una variable no puede repetirse dentro del antecedente.")

    activaciones = {
        condicion["variable"]: calcular_activacion_condicion(
            dataframe,
            condicion,
            metodo_similitud
        )
        for condicion in condiciones
    }

    activacion_conjunta = np.minimum.reduce(list(activaciones.values()))

    if devolver_detalle:
        return activacion_conjunta, activaciones

    return activacion_conjunta

In [ ]:
# ============================================================
# RECONSTRUIR UNA REGLA DOMINANTE
# ============================================================

def calcular_factor_certeza(confianza, soporte_consecuente):
    """Calcula el factor de certeza de una regla."""
    if confianza > soporte_consecuente:
        return (confianza - soporte_consecuente) / (1.0 - soporte_consecuente)

    if confianza < soporte_consecuente:
        return (confianza - soporte_consecuente) / soporte_consecuente

    return 0.0


def calcular_metricas_dominante(activacion_X, objetivo, valor_consecuente):
    """Recalcula las métricas de una regla dominante."""
    activacion_Y = (np.asarray(objetivo) == valor_consecuente).astype(float)
    activacion_XY = np.minimum(activacion_X, activacion_Y)

    masa_X = activacion_X.sum()
    soporte_X = activacion_X.mean()
    soporte_XY = activacion_XY.mean()
    soporte_Y = activacion_Y.mean()
    confianza = activacion_XY.sum() / masa_X if masa_X > 0 else 0.0
    lift = confianza / soporte_Y if soporte_Y > 0 else 0.0

    return {
        "soporte_X": soporte_X,
        "soporte_XY": soporte_XY,
        "confianza": confianza,
        "lift": lift,
        "factor_certeza": calcular_factor_certeza(confianza, soporte_Y)
    }


# 1. Preparar antecedentes y objetivo
columnas_requeridas = columnas_antecedente + [columna_objetivo]

if df_similitud[columnas_requeridas].isna().any().any():
    raise ValueError("El dataset contiene valores faltantes.")

X_datos = df_similitud[columnas_antecedente].copy()
y_objetivo = df_similitud[columna_objetivo].astype(np.int8)

# 2. Seleccionar la primera regla como referencia
indice_referencia = 0
fila_referencia = catalogo_dominantes_principal.iloc[indice_referencia]
regla_referencia = reglas_dominantes_principales[indice_referencia]
resultado_referencia = regla_referencia["resultado"]

condiciones_X = resultado_referencia["condiciones"]
metricas_guardadas = resultado_referencia["metricas"]

valor_consecuente = int(
    regla_referencia.get(
        "valor_consecuente",
        metricas_guardadas["valor_consecuente"]
    )
)

metodo_similitud = regla_referencia.get(
    "metodo_similitud",
    metricas_guardadas["metodo_similitud"]
)

valor_consecuente_opuesto = 1 - valor_consecuente
variables_X = [condicion["variable"] for condicion in condiciones_X]
variables_disponibles_Z = [
    variable for variable in columnas_antecedente
    if variable not in variables_X
]

# 3. Reconstruir activación y métricas
activacion_X = calcular_activacion_antecedente(
    X_datos,
    condiciones_X,
    metodo_similitud
)

metricas_recalculadas = calcular_metricas_dominante(
    activacion_X,
    y_objetivo,
    valor_consecuente
)

# 4. Comparar con las métricas guardadas
nombres_metricas = [
    "soporte_X",
    "soporte_XY",
    "confianza",
    "lift",
    "factor_certeza"
]

tabla_verificacion = pd.DataFrame({
    "metrica": nombres_metricas,
    "guardada": [metricas_guardadas[nombre] for nombre in nombres_metricas],
    "recalculada": [metricas_recalculadas[nombre] for nombre in nombres_metricas]
})

tabla_verificacion["diferencia"] = np.abs(
    tabla_verificacion["guardada"] - tabla_verificacion["recalculada"]
)

tabla_verificacion["coincide"] = np.isclose(
    tabla_verificacion["guardada"],
    tabla_verificacion["recalculada"],
    atol=1e-6
)

# 5. Mostrar resultado
print(
    f"Regla: {fila_referencia['regla']}\n"
    f"Método: {metodo_similitud}\n"
    f"Consecuente dominante: {valor_consecuente}\n"
    f"Consecuente anómalo: {valor_consecuente_opuesto}\n"
    f"Variables de X: {variables_X}\n"
    f"Variables disponibles para Z: {variables_disponibles_Z}"
)

display(tabla_verificacion.round(10))

In [ ]:
# ============================================================
# VALIDAR LA CONDICIÓN Z
# ============================================================

valores_categoricos = {
    columna: sorted(X_datos[columna].unique().tolist())
    for columna in columnas_categoricas
}


def verificar_validez_Z(
    condiciones_Z,
    variables_X,
    radio_minimo,
    radio_maximo,
    min_condiciones=1,
    max_condiciones=3
):
    """Verifica que Z sea válida respecto a una regla dominante."""
    errores = []

    if not min_condiciones <= len(condiciones_Z) <= max_condiciones:
        errores.append(
            f"Z debe contener entre {min_condiciones} y "
            f"{max_condiciones} condiciones."
        )

    variables_Z = [condicion["variable"] for condicion in condiciones_Z]

    if len(variables_Z) != len(set(variables_Z)):
        errores.append("Z contiene variables repetidas.")

    variables_reutilizadas = set(variables_Z).intersection(variables_X)

    if variables_reutilizadas:
        errores.append(
            f"Z reutiliza variables de X: {sorted(variables_reutilizadas)}"
        )

    for condicion in condiciones_Z:
        variable = condicion["variable"]
        tipo = condicion["tipo"]

        if variable not in columnas_antecedente:
            errores.append(f"Variable desconocida en Z: {variable}")
            continue

        if variable in columnas_continuas:
            if tipo != "continua":
                errores.append(f"'{variable}' debe ser continua.")
                continue

            centro = float(condicion["centro"])
            radio = float(condicion["radio"])

            if not 0.0 <= centro <= 1.0:
                errores.append(f"Centro fuera de [0, 1] en '{variable}'.")

            if not radio_minimo <= radio <= radio_maximo:
                errores.append(
                    f"Radio inválido en '{variable}': {radio}"
                )

        elif variable in columnas_categoricas:
            if tipo != "categorica":
                errores.append(f"'{variable}' debe ser categórica.")
                continue

            if condicion["valor"] not in valores_categoricos[variable]:
                errores.append(
                    f"Valor categórico inválido en '{variable}': "
                    f"{condicion['valor']}"
                )

    return {
        "es_valida": len(errores) == 0,
        "compuerta": 1.0 if not errores else 0.0,
        "errores": errores,
        "variables_Z": variables_Z
    }

## 5. Metricas de una candidata anomala

Para una regla dominante $R_d:X\Rightarrow Y$, el segundo PSO mantiene $X$ fijo y evalua solamente las condiciones adicionales $Z$. La candidata completa es $R_a:X\land Z\Rightarrow\neg Y$.

La metrica principal es el soporte anomaloso $Supp_a$. Las dos confianzas condicionadas permiten comprobar tanto la presencia de $\neg Y$ dentro de $X\land Z$ como la concentracion de $Z$ entre los fallos de $R_d$ dentro de $X$.


In [ ]:
# ============================================================
# CALCULAR MÉTRICAS DE ANOMALÍA
# ============================================================

def calcular_metricas_anomalia(
    activacion_X,
    activacion_Z,
    objetivo,
    valor_consecuente_dominante,
    devolver_activaciones=False
):
    """Calcula las métricas de Z dentro del contexto dominante X."""
    activacion_X = np.asarray(activacion_X, dtype=float)
    activacion_Z = np.asarray(activacion_Z, dtype=float)
    objetivo = np.asarray(objetivo)

    if len(activacion_X) != len(activacion_Z) or len(activacion_X) != len(objetivo):
        raise ValueError("Las activaciones y el objetivo deben tener la misma longitud.")

    valor_anomalo = 1 - int(valor_consecuente_dominante)
    activacion_noY = (objetivo == valor_anomalo).astype(float)

    activacion_XZ = np.minimum(activacion_X, activacion_Z)
    activacion_X_noY = np.minimum(activacion_X, activacion_noY)
    activacion_XZ_noY = np.minimum(activacion_XZ, activacion_noY)

    masa_X = activacion_X.sum()
    masa_XZ = activacion_XZ.sum()
    masa_X_noY = activacion_X_noY.sum()
    masa_XZ_noY = activacion_XZ_noY.sum()
    numero_registros = len(objetivo)

    soporte_X = masa_X / numero_registros
    soporte_XZ = masa_XZ / numero_registros
    soporte_anomalo = masa_XZ_noY / numero_registros

    confianza_anomala = masa_XZ_noY / masa_XZ if masa_XZ > 0 else 0.0
    confianza_inversa = masa_XZ_noY / masa_X_noY if masa_X_noY > 0 else 0.0

    base_noY_en_X = masa_X_noY / masa_X if masa_X > 0 else 0.0
    base_Z_en_X = masa_XZ / masa_X if masa_X > 0 else 0.0

    cf_Z_hacia_noY = calcular_factor_certeza(
        confianza_anomala,
        base_noY_en_X
    )

    cf_noY_hacia_Z = calcular_factor_certeza(
        confianza_inversa,
        base_Z_en_X
    )

    lift_condicional = (
        confianza_anomala / base_noY_en_X
        if base_noY_en_X > 0 else 0.0
    )

    resultado = {
        "valor_consecuente_anomalo": valor_anomalo,
        "soporte_X": soporte_X,
        "soporte_XZ": soporte_XZ,
        "soporte_anomalo": soporte_anomalo,
        "confianza_anomala": confianza_anomala,
        "confianza_inversa": confianza_inversa,
        "base_noY_en_X": base_noY_en_X,
        "base_Z_en_X": base_Z_en_X,
        "lift_condicional": lift_condicional,
        "cf_Z_hacia_noY": cf_Z_hacia_noY,
        "cf_noY_hacia_Z": cf_noY_hacia_Z,
        "masa_X": masa_X,
        "masa_XZ": masa_XZ,
        "masa_X_noY": masa_X_noY,
        "masa_XZ_noY": masa_XZ_noY,
        "registros_XZ_positivos": int(np.count_nonzero(activacion_XZ)),
        "registros_anomalos_positivos": int(np.count_nonzero(activacion_XZ_noY))
    }

    if devolver_activaciones:
        resultado["activaciones"] = {
            "X": activacion_X,
            "Z": activacion_Z,
            "XZ": activacion_XZ,
            "X_noY": activacion_X_noY,
            "XZ_noY": activacion_XZ_noY
        }

    return resultado

In [ ]:
# ============================================================
# EVALUAR UNA REGLA ANÓMALA
# ============================================================

def calcular_fitness_anomalia(
    metricas,
    longitud_Z,
    soporte_minimo,
    compuerta=1.0
):
    """Calcula el fitness anómalo y devuelve sus componentes."""
    if soporte_minimo <= 0:
        raise ValueError("El soporte mínimo debe ser mayor que cero.")

    componente_cf_directo = max(0.0, metricas["cf_Z_hacia_noY"])
    componente_cf_inverso = max(0.0, metricas["cf_noY_hacia_Z"])
    componente_soporte = min(
        1.0,
        max(0.0, metricas["soporte_anomalo"] / soporte_minimo)
    )
    componente_longitud = 1.0 / (1.0 + longitud_Z)

    fitness = (
        compuerta
        * componente_cf_directo
        * componente_cf_inverso
        * componente_soporte
        * componente_longitud
    )

    return {
        "compuerta": compuerta,
        "cf_directo": componente_cf_directo,
        "cf_inverso": componente_cf_inverso,
        "soporte": componente_soporte,
        "longitud": componente_longitud,
        "fitness": fitness
    }


def evaluar_regla_anomala(
    dataframe,
    objetivo,
    activacion_X,
    condiciones_Z,
    variables_X,
    valor_consecuente_dominante,
    metodo_similitud,
    soporte_minimo,
    radio_minimo,
    radio_maximo
):
    """Evalúa una candidata X y Z que implica el consecuente contrario."""
    validez = verificar_validez_Z(
        condiciones_Z=condiciones_Z,
        variables_X=variables_X,
        radio_minimo=radio_minimo,
        radio_maximo=radio_maximo
    )

    resultado_base = {
        "condiciones_Z": condiciones_Z,
        "valor_consecuente_dominante": valor_consecuente_dominante,
        "valor_consecuente_anomalo": 1 - valor_consecuente_dominante,
        "metodo_similitud": metodo_similitud,
        "validez": validez
    }

    if not validez["es_valida"]:
        return {
            **resultado_base,
            "metricas": None,
            "componentes_fitness": {
                "compuerta": 0.0,
                "cf_directo": 0.0,
                "cf_inverso": 0.0,
                "soporte": 0.0,
                "longitud": 0.0,
                "fitness": 0.0
            },
            "fitness": 0.0
        }

    activacion_Z = calcular_activacion_antecedente(
        dataframe,
        condiciones_Z,
        metodo_similitud
    )

    metricas = calcular_metricas_anomalia(
        activacion_X=activacion_X,
        activacion_Z=activacion_Z,
        objetivo=objetivo,
        valor_consecuente_dominante=valor_consecuente_dominante
    )

    componentes = calcular_fitness_anomalia(
        metricas=metricas,
        longitud_Z=len(condiciones_Z),
        soporte_minimo=soporte_minimo,
        compuerta=validez["compuerta"]
    )

    return {
        **resultado_base,
        "metricas": metricas,
        "componentes_fitness": componentes,
        "fitness": componentes["fitness"]
    }

In [ ]:
# ============================================================
# DEFINIR CRITERIO DE ANOMALÍA
# ============================================================

MIN_CONDICIONES_Z = 1
MAX_CONDICIONES_Z = 3

SOPORTES_ANOMALOS = [0.0005, 0.001, 0.002]
CONFIANZAS_ANOMALAS = [0.60, 0.70, 0.80]


def verificar_regla_anomala(
    resultado_regla,
    soporte_minimo,
    confianza_minima,
    min_condiciones=1,
    max_condiciones=3
):
    """Verifica si una candidata cumple los criterios de anomalía."""
    metricas = resultado_regla.get("metricas")
    estructura_valida = resultado_regla["validez"]["es_valida"]
    longitud_Z = len(resultado_regla["condiciones_Z"])

    criterios = {
        "estructura_valida": estructura_valida,
        "fitness_positivo": resultado_regla["fitness"] > 0,
        "soporte_suficiente": (
            metricas is not None
            and metricas["soporte_anomalo"] >= soporte_minimo
        ),
        "confianza_suficiente": (
            metricas is not None
            and metricas["confianza_anomala"] >= confianza_minima
        ),
        "cf_directo_positivo": (
            metricas is not None
            and metricas["cf_Z_hacia_noY"] > 0
        ),
        "cf_inverso_positivo": (
            metricas is not None
            and metricas["cf_noY_hacia_Z"] > 0
        ),
        "longitud_valida": min_condiciones <= longitud_Z <= max_condiciones
    }

    criterios_incumplidos = [
        nombre for nombre, cumple in criterios.items()
        if not cumple
    ]

    return {
        "es_anomala": all(criterios.values()),
        "criterios": criterios,
        "criterios_incumplidos": criterios_incumplidos,
        "umbrales": {
            "soporte_minimo": soporte_minimo,
            "confianza_minima": confianza_minima,
            "min_condiciones": min_condiciones,
            "max_condiciones": max_condiciones
        }
    }


# Crear las nueve combinaciones que se compararán
configuraciones_umbrales = [
    {
        "configuracion": f"supp_{soporte:g}_conf_{confianza:.2f}",
        "soporte_minimo": soporte,
        "confianza_minima": confianza,
        "masa_equivalente_minima": soporte * len(df_similitud)
    }
    for soporte, confianza in product(
        SOPORTES_ANOMALOS,
        CONFIANZAS_ANOMALAS
    )
]

tabla_configuraciones_umbrales = pd.DataFrame(configuraciones_umbrales)

display(
    tabla_configuraciones_umbrales.round({
        "soporte_minimo": 4,
        "confianza_minima": 2,
        "masa_equivalente_minima": 1
    })
)

## 7. Representacion de la particula $Z$

Cada particula del segundo PSO representa unicamente $Z$. Las variables que ya pertenecen a $X$ se excluyen del espacio de busqueda. Las variables continuas se actualizan mediante centro y radio; las categoricas se seleccionan entre el valor actual, `pBest`, `gBest` o una categoria exploratoria.


In [ ]:
# ============================================================
# REPRESENTAR UNA PARTÍCULA Z
# ============================================================

LIMITE_CENTRO_INFERIOR = 0.0
LIMITE_CENTRO_SUPERIOR = 1.0

RADIO_MINIMO_Z = 0.02
RADIO_MAXIMO_Z = 0.30


def crear_particula_Z_vacia(variables_X):
    """Crea una partícula Z sin condiciones activas."""
    variables_X = set(variables_X)
    particula = {"continuas": {}, "categoricas": {}}

    for variable in columnas_continuas:
        if variable not in variables_X:
            particula["continuas"][variable] = {
                "activa": 0,
                "centro": 0.5,
                "radio": 0.1
            }

    for variable in columnas_categoricas:
        if variable not in variables_X:
            particula["categoricas"][variable] = {
                "activa": 0,
                "valor": valores_categoricos[variable][0]
            }

    return particula


def contar_condiciones_Z(particula):
    """Cuenta las condiciones activas de Z."""
    continuas = sum(
        dato["activa"] for dato in particula["continuas"].values()
    )
    categoricas = sum(
        dato["activa"] for dato in particula["categoricas"].values()
    )

    return int(continuas + categoricas)


def decodificar_particula_Z(particula):
    """Convierte una partícula Z en una lista de condiciones."""
    condiciones_Z = []

    for variable, dato in particula["continuas"].items():
        if dato["activa"] == 1:
            condiciones_Z.append({
                "variable": variable,
                "tipo": "continua",
                "centro": float(dato["centro"]),
                "radio": float(dato["radio"])
            })

    for variable, dato in particula["categoricas"].items():
        if dato["activa"] == 1:
            condiciones_Z.append({
                "variable": variable,
                "tipo": "categorica",
                "valor": dato["valor"]
            })

    return condiciones_Z

In [ ]:
# ============================================================
# GENERAR PARTÍCULAS Z ALEATORIAS
# ============================================================

def crear_particula_Z_aleatoria(
    generador,
    variables_X,
    min_condiciones=MIN_CONDICIONES_Z,
    max_condiciones=MAX_CONDICIONES_Z,
    radio_minimo=RADIO_MINIMO_Z,
    radio_maximo=RADIO_MAXIMO_Z
):
    """Crea una partícula Z aleatoria y estructuralmente válida."""
    particula = crear_particula_Z_vacia(variables_X)

    variables_disponibles = (
        list(particula["continuas"])
        + list(particula["categoricas"])
    )

    if min_condiciones < 1:
        raise ValueError("Z debe contener al menos una condición.")

    if max_condiciones < min_condiciones:
        raise ValueError("El máximo de condiciones no puede ser menor que el mínimo.")

    if max_condiciones > len(variables_disponibles):
        raise ValueError("No existen suficientes variables disponibles para formar Z.")

    if radio_minimo <= 0 or radio_maximo <= radio_minimo:
        raise ValueError("Los límites del radio no son válidos.")

    numero_condiciones = int(
        generador.integers(min_condiciones, max_condiciones + 1)
    )

    variables_seleccionadas = generador.choice(
        variables_disponibles,
        size=numero_condiciones,
        replace=False
    )

    for variable in variables_seleccionadas:
        variable = str(variable)

        if variable in particula["continuas"]:
            particula["continuas"][variable] = {
                "activa": 1,
                "centro": float(generador.uniform(0.0, 1.0)),
                "radio": float(generador.uniform(radio_minimo, radio_maximo))
            }

        else:
            valor = generador.choice(valores_categoricos[variable])

            if hasattr(valor, "item"):
                valor = valor.item()

            particula["categoricas"][variable] = {
                "activa": 1,
                "valor": valor
            }

    return particula

In [ ]:
EJECUTAR_PRUEBA_PARTICULA_Z = False

if EJECUTAR_PRUEBA_PARTICULA_Z:
    # ============================================================
    # PROBAR UNA PARTÍCULA Z
    # ============================================================

    generador_prueba = np.random.default_rng(2026)

    particula_Z_prueba = crear_particula_Z_aleatoria(
        generador=generador_prueba,
        variables_X=variables_X
    )

    condiciones_Z_prueba = decodificar_particula_Z(particula_Z_prueba)
    resultados_prueba_soporte = {}
    filas_prueba = []

    for soporte_minimo in SOPORTES_ANOMALOS:
        resultado = evaluar_regla_anomala(
            dataframe=X_datos,
            objetivo=y_objetivo,
            activacion_X=activacion_X,
            condiciones_Z=condiciones_Z_prueba,
            variables_X=variables_X,
            valor_consecuente_dominante=valor_consecuente,
            metodo_similitud=metodo_similitud,
            soporte_minimo=soporte_minimo,
            radio_minimo=RADIO_MINIMO_Z,
            radio_maximo=RADIO_MAXIMO_Z
        )

        resultados_prueba_soporte[soporte_minimo] = resultado
        metricas = resultado["metricas"]
        componentes = resultado["componentes_fitness"]

        for confianza_minima in CONFIANZAS_ANOMALAS:
            validacion = verificar_regla_anomala(
                resultado_regla=resultado,
                soporte_minimo=soporte_minimo,
                confianza_minima=confianza_minima,
                min_condiciones=MIN_CONDICIONES_Z,
                max_condiciones=MAX_CONDICIONES_Z
            )

            filas_prueba.append({
                "min_supp": soporte_minimo,
                "min_conf": confianza_minima,
                "fitness": resultado["fitness"],
                "soporte_anomalo": metricas["soporte_anomalo"],
                "confianza_anomala": metricas["confianza_anomala"],
                "confianza_inversa": metricas["confianza_inversa"],
                "cf_directo": metricas["cf_Z_hacia_noY"],
                "cf_inverso": metricas["cf_noY_hacia_Z"],
                "componente_soporte": componentes["soporte"],
                "componente_longitud": componentes["longitud"],
                "es_anomala": validacion["es_anomala"],
                "criterios_incumplidos": (
                    ", ".join(validacion["criterios_incumplidos"])
                    or "ninguno"
                )
            })

    tabla_prueba_particula = pd.DataFrame(filas_prueba)

    print(
        f"Regla dominante:\n{fila_referencia['regla']}\n\n"
        f"Consecuente anómalo: {valor_consecuente_opuesto}\n"
        f"Método de similaridad: {metodo_similitud}\n"
        f"Condiciones de Z: {len(condiciones_Z_prueba)}"
    )

    display(pd.DataFrame(condiciones_Z_prueba))

    display(
        tabla_prueba_particula.round({
            "min_supp": 4,
            "min_conf": 2,
            "fitness": 8,
            "soporte_anomalo": 6,
            "confianza_anomala": 6,
            "confianza_inversa": 6,
            "cf_directo": 6,
            "cf_inverso": 6,
            "componente_soporte": 6,
            "componente_longitud": 6
        })
    )
else:
    generador_prueba = None
    particula_Z_prueba = None
    condiciones_Z_prueba = []
    resultados_prueba_soporte = {}
    tabla_prueba_particula = pd.DataFrame()
    print("Prueba de particula Z omitida; el experimento principal no se modifica.")


In [ ]:
# ============================================================
# CREAR VELOCIDADES INICIALES
# ============================================================

RANGO_VELOCIDAD_ACTIVACION_INICIAL = 1.0
RANGO_VELOCIDAD_CENTRO_INICIAL = 0.05
RANGO_VELOCIDAD_RADIO_INICIAL = 0.02


def crear_velocidad_Z_inicial(generador, particula):
    """Crea las velocidades iniciales de una partícula Z."""
    velocidad = {"activacion": {}, "continuas": {}}

    variables_disponibles = (
        list(particula["continuas"])
        + list(particula["categoricas"])
    )

    for variable in variables_disponibles:
        velocidad["activacion"][variable] = float(
            generador.uniform(
                -RANGO_VELOCIDAD_ACTIVACION_INICIAL,
                RANGO_VELOCIDAD_ACTIVACION_INICIAL
            )
        )

    for variable in particula["continuas"]:
        velocidad["continuas"][variable] = {
            "centro": float(
                generador.uniform(
                    -RANGO_VELOCIDAD_CENTRO_INICIAL,
                    RANGO_VELOCIDAD_CENTRO_INICIAL
                )
            ),
            "radio": float(
                generador.uniform(
                    -RANGO_VELOCIDAD_RADIO_INICIAL,
                    RANGO_VELOCIDAD_RADIO_INICIAL
                )
            )
        }

    return velocidad

In [ ]:
# ============================================================
# ACTUALIZAR VELOCIDADES
# ============================================================

W_INERCIA = 0.70
C1_COGNITIVO = 1.50
C2_SOCIAL = 1.50

VELOCIDAD_MAX_ACTIVACION = 4.0
VELOCIDAD_MAX_CENTRO = 0.20
VELOCIDAD_MAX_RADIO = 0.10


def obtener_estado_activacion_Z(particula, variable):
    """Devuelve 1 si una variable está activa en Z y 0 en caso contrario."""
    if variable in particula["continuas"]:
        return int(particula["continuas"][variable]["activa"])

    if variable in particula["categoricas"]:
        return int(particula["categoricas"][variable]["activa"])

    raise ValueError(f"La variable '{variable}' no pertenece a la partícula.")


def actualizar_dimension_pso(
    posicion_actual,
    velocidad_actual,
    posicion_pbest,
    posicion_gbest,
    generador,
    limite,
    w,
    c1,
    c2
):
    """Actualiza una dimensión mediante la ecuación de PSO."""
    velocidad = (
        w * velocidad_actual
        + c1 * generador.random() * (posicion_pbest - posicion_actual)
        + c2 * generador.random() * (posicion_gbest - posicion_actual)
    )

    return float(np.clip(velocidad, -limite, limite))


def actualizar_velocidad_Z(
    particula_actual,
    velocidad_actual,
    pbest_particula,
    gbest_particula,
    generador,
    w=W_INERCIA,
    c1=C1_COGNITIVO,
    c2=C2_SOCIAL
):
    """Actualiza las velocidades de activación, centro y radio."""
    nueva_velocidad = copy.deepcopy(velocidad_actual)

    for variable in velocidad_actual["activacion"]:
        nueva_velocidad["activacion"][variable] = actualizar_dimension_pso(
            posicion_actual=obtener_estado_activacion_Z(
                particula_actual,
                variable
            ),
            velocidad_actual=velocidad_actual["activacion"][variable],
            posicion_pbest=obtener_estado_activacion_Z(
                pbest_particula,
                variable
            ),
            posicion_gbest=obtener_estado_activacion_Z(
                gbest_particula,
                variable
            ),
            generador=generador,
            limite=VELOCIDAD_MAX_ACTIVACION,
            w=w,
            c1=c1,
            c2=c2
        )

    limites_continuos = {
        "centro": VELOCIDAD_MAX_CENTRO,
        "radio": VELOCIDAD_MAX_RADIO
    }

    for variable in particula_actual["continuas"]:
        for parametro, limite in limites_continuos.items():
            nueva_velocidad["continuas"][variable][parametro] = (
                actualizar_dimension_pso(
                    posicion_actual=particula_actual[
                        "continuas"
                    ][variable][parametro],
                    velocidad_actual=velocidad_actual[
                        "continuas"
                    ][variable][parametro],
                    posicion_pbest=pbest_particula[
                        "continuas"
                    ][variable][parametro],
                    posicion_gbest=gbest_particula[
                        "continuas"
                    ][variable][parametro],
                    generador=generador,
                    limite=limite,
                    w=w,
                    c1=c1,
                    c2=c2
                )
            )

    return nueva_velocidad

In [ ]:
# ============================================================
# ACTUALIZAR POSICIÓN DE UNA PARTÍCULA Z
# ============================================================

PROBABILIDAD_EXPLORACION_CATEGORICA = 0.10


def funcion_sigmoide(velocidad):
    """Convierte una velocidad de activación en probabilidad."""
    velocidad = np.clip(velocidad, -60.0, 60.0)
    return float(1.0 / (1.0 + np.exp(-velocidad)))


def establecer_activacion_Z(particula, variable, estado):
    """Activa o desactiva una variable de Z."""
    if estado not in {0, 1}:
        raise ValueError("El estado de activación debe ser 0 o 1.")

    if variable in particula["continuas"]:
        grupo = "continuas"
    elif variable in particula["categoricas"]:
        grupo = "categoricas"
    else:
        raise ValueError(f"La variable '{variable}' no pertenece a Z.")

    particula[grupo][variable]["activa"] = int(estado)


def actualizar_activaciones_Z(
    particula,
    velocidad,
    generador,
    min_condiciones=MIN_CONDICIONES_Z,
    max_condiciones=MAX_CONDICIONES_Z
):
    """Actualiza las activaciones y repara la longitud de Z."""
    nueva_particula = copy.deepcopy(particula)
    probabilidades = {}

    for variable, velocidad_variable in velocidad["activacion"].items():
        probabilidad = funcion_sigmoide(velocidad_variable)
        estado = int(generador.random() < probabilidad)

        establecer_activacion_Z(nueva_particula, variable, estado)
        probabilidades[variable] = probabilidad

    variables_activas = [
        variable for variable in probabilidades
        if obtener_estado_activacion_Z(nueva_particula, variable) == 1
    ]

    if len(variables_activas) < min_condiciones:
        variables_ordenadas = sorted(
            probabilidades,
            key=probabilidades.get,
            reverse=True
        )

        for variable in variables_ordenadas[:min_condiciones]:
            establecer_activacion_Z(nueva_particula, variable, 1)

    elif len(variables_activas) > max_condiciones:
        variables_conservar = set(
            sorted(
                variables_activas,
                key=probabilidades.get,
                reverse=True
            )[:max_condiciones]
        )

        for variable in variables_activas:
            establecer_activacion_Z(
                nueva_particula,
                variable,
                int(variable in variables_conservar)
            )

    return nueva_particula


def actualizar_continuas_Z(particula, velocidad):
    """Actualiza centros y radios de las variables continuas."""
    nueva_particula = copy.deepcopy(particula)

    for variable, datos in particula["continuas"].items():
        centro = datos["centro"] + velocidad["continuas"][variable]["centro"]
        radio = datos["radio"] + velocidad["continuas"][variable]["radio"]

        nueva_particula["continuas"][variable]["centro"] = float(
            np.clip(
                centro,
                LIMITE_CENTRO_INFERIOR,
                LIMITE_CENTRO_SUPERIOR
            )
        )

        nueva_particula["continuas"][variable]["radio"] = float(
            np.clip(
                radio,
                RADIO_MINIMO_Z,
                RADIO_MAXIMO_Z
            )
        )

    return nueva_particula


def actualizar_categorias_Z(
    particula,
    pbest_particula,
    gbest_particula,
    generador,
    probabilidad_exploracion=PROBABILIDAD_EXPLORACION_CATEGORICA
):
    """Actualiza los valores categóricos mediante copia o exploración."""
    if not 0.0 <= probabilidad_exploracion <= 1.0:
        raise ValueError("La probabilidad de exploración debe estar entre 0 y 1.")

    nueva_particula = copy.deepcopy(particula)

    for variable, datos in particula["categoricas"].items():
        if generador.random() < probabilidad_exploracion:
            valor_nuevo = generador.choice(valores_categoricos[variable])
        else:
            pesos = np.array([
                W_INERCIA,
                C1_COGNITIVO * generador.random(),
                C2_SOCIAL * generador.random()
            ])

            probabilidades = pesos / pesos.sum()
            fuente = int(generador.choice(3, p=probabilidades))

            valores = [
                datos["valor"],
                pbest_particula["categoricas"][variable]["valor"],
                gbest_particula["categoricas"][variable]["valor"]
            ]

            valor_nuevo = valores[fuente]

        if hasattr(valor_nuevo, "item"):
            valor_nuevo = valor_nuevo.item()

        nueva_particula["categoricas"][variable]["valor"] = valor_nuevo

    return nueva_particula


def actualizar_posicion_Z(
    particula,
    velocidad,
    pbest_particula,
    gbest_particula,
    generador,
    probabilidad_exploracion=PROBABILIDAD_EXPLORACION_CATEGORICA
):
    """Ejecuta la actualización completa de una partícula Z."""
    nueva_particula = actualizar_activaciones_Z(
        particula,
        velocidad,
        generador
    )

    nueva_particula = actualizar_continuas_Z(
        nueva_particula,
        velocidad
    )

    nueva_particula = actualizar_categorias_Z(
        nueva_particula,
        pbest_particula,
        gbest_particula,
        generador,
        probabilidad_exploracion
    )

    return nueva_particula

In [ ]:
# ============================================================
# EJECUTAR UNA ITERACIÓN DE PSO ANÓMALO
# ============================================================

def evaluar_particula_Z(
    particula,
    dataframe,
    objetivo,
    activacion_X,
    variables_X,
    valor_consecuente_dominante,
    metodo_similitud,
    soporte_minimo
):
    """Decodifica y evalúa una partícula Z."""
    condiciones_Z = decodificar_particula_Z(particula)

    return evaluar_regla_anomala(
        dataframe=dataframe,
        objetivo=objetivo,
        activacion_X=activacion_X,
        condiciones_Z=condiciones_Z,
        variables_X=variables_X,
        valor_consecuente_dominante=valor_consecuente_dominante,
        metodo_similitud=metodo_similitud,
        soporte_minimo=soporte_minimo,
        radio_minimo=RADIO_MINIMO_Z,
        radio_maximo=RADIO_MAXIMO_Z
    )


def ejecutar_iteracion_pso_anomalo(
    enjambre,
    velocidades,
    pbest_particulas,
    pbest_fitness,
    pbest_resultados,
    gbest_particula,
    gbest_fitness,
    gbest_resultado,
    dataframe,
    objetivo,
    activacion_X,
    variables_X,
    valor_consecuente_dominante,
    metodo_similitud,
    soporte_minimo,
    generador,
    probabilidad_exploracion=PROBABILIDAD_EXPLORACION_CATEGORICA
):
    """Ejecuta una iteración completa del PSO anómalo."""
    numero_particulas = len(enjambre)

    if not (
        len(velocidades)
        == len(pbest_particulas)
        == len(pbest_fitness)
        == len(pbest_resultados)
        == numero_particulas
    ):
        raise ValueError("Las estructuras del enjambre tienen tamaños diferentes.")

    gbest_inicio = copy.deepcopy(gbest_particula)
    gbest_fitness_inicio = float(gbest_fitness)

    nuevo_enjambre = []
    nuevas_velocidades = []
    nuevos_resultados = []

    nuevos_pbest_particulas = copy.deepcopy(pbest_particulas)
    nuevos_pbest_fitness = np.asarray(pbest_fitness, dtype=float).copy()
    nuevos_pbest_resultados = copy.deepcopy(pbest_resultados)

    fitness_actuales = []
    pbest_actualizados = 0

    for indice, particula_actual in enumerate(enjambre):
        velocidad_nueva = actualizar_velocidad_Z(
            particula_actual=particula_actual,
            velocidad_actual=velocidades[indice],
            pbest_particula=pbest_particulas[indice],
            gbest_particula=gbest_inicio,
            generador=generador
        )

        particula_nueva = actualizar_posicion_Z(
            particula=particula_actual,
            velocidad=velocidad_nueva,
            pbest_particula=pbest_particulas[indice],
            gbest_particula=gbest_inicio,
            generador=generador,
            probabilidad_exploracion=probabilidad_exploracion
        )

        resultado_nuevo = evaluar_particula_Z(
            particula=particula_nueva,
            dataframe=dataframe,
            objetivo=objetivo,
            activacion_X=activacion_X,
            variables_X=variables_X,
            valor_consecuente_dominante=valor_consecuente_dominante,
            metodo_similitud=metodo_similitud,
            soporte_minimo=soporte_minimo
        )

        fitness_nuevo = resultado_nuevo["fitness"]
        fitness_actuales.append(fitness_nuevo)

        if fitness_nuevo > nuevos_pbest_fitness[indice]:
            nuevos_pbest_particulas[indice] = copy.deepcopy(particula_nueva)
            nuevos_pbest_fitness[indice] = fitness_nuevo
            nuevos_pbest_resultados[indice] = copy.deepcopy(resultado_nuevo)
            pbest_actualizados += 1

        nuevo_enjambre.append(particula_nueva)
        nuevas_velocidades.append(velocidad_nueva)
        nuevos_resultados.append(resultado_nuevo)

    indice_gbest = int(np.argmax(nuevos_pbest_fitness))
    nuevo_gbest_particula = copy.deepcopy(
        nuevos_pbest_particulas[indice_gbest]
    )
    nuevo_gbest_fitness = float(
        nuevos_pbest_fitness[indice_gbest]
    )
    nuevo_gbest_resultado = copy.deepcopy(
        nuevos_pbest_resultados[indice_gbest]
    )

    metricas_gbest = nuevo_gbest_resultado["metricas"]

    resumen = {
        "gbest_fitness": nuevo_gbest_fitness,
        "mejora_gbest": nuevo_gbest_fitness > gbest_fitness_inicio,
        "fitness_promedio_actual": float(np.mean(fitness_actuales)),
        "fitness_maximo_actual": float(np.max(fitness_actuales)),
        "particulas_fitness_positivo": int(
            np.count_nonzero(np.asarray(fitness_actuales) > 0)
        ),
        "pbest_actualizados": pbest_actualizados,
        "numero_condiciones_gbest": len(
            nuevo_gbest_resultado["condiciones_Z"]
        ),
        "soporte_anomalo_gbest": metricas_gbest["soporte_anomalo"],
        "confianza_anomala_gbest": metricas_gbest["confianza_anomala"],
        "cf_directo_gbest": metricas_gbest["cf_Z_hacia_noY"],
        "cf_inverso_gbest": metricas_gbest["cf_noY_hacia_Z"]
    }

    return {
        "enjambre": nuevo_enjambre,
        "velocidades": nuevas_velocidades,
        "resultados": nuevos_resultados,
        "pbest_particulas": nuevos_pbest_particulas,
        "pbest_fitness": nuevos_pbest_fitness,
        "pbest_resultados": nuevos_pbest_resultados,
        "gbest_particula": nuevo_gbest_particula,
        "gbest_fitness": nuevo_gbest_fitness,
        "gbest_resultado": nuevo_gbest_resultado,
        "resumen": resumen
    }

In [ ]:
# ============================================================
# EJECUTAR PSO PARA UNA REGLA DOMINANTE
# ============================================================

def ejecutar_pso_anomalo(
    dataframe,
    objetivo,
    activacion_X,
    variables_X,
    valor_consecuente_dominante,
    metodo_similitud,
    soporte_minimo,
    numero_particulas=50,
    numero_iteraciones=50,
    probabilidad_exploracion=PROBABILIDAD_EXPLORACION_CATEGORICA,
    semilla=55
):
    """Ejecuta el PSO anómalo para una regla dominante."""
    if numero_particulas < 2:
        raise ValueError("El número de partículas debe ser al menos 2.")

    if numero_iteraciones < 1:
        raise ValueError("El número de iteraciones debe ser al menos 1.")

    if valor_consecuente_dominante not in {0, 1}:
        raise ValueError("El consecuente dominante debe ser 0 o 1.")

    if metodo_similitud not in {"triangular", "gaussiana"}:
        raise ValueError("Método de similaridad no válido.")

    if soporte_minimo <= 0:
        raise ValueError("El soporte mínimo debe ser mayor que cero.")

    generador = np.random.default_rng(semilla)

    enjambre = [
        crear_particula_Z_aleatoria(
            generador=generador,
            variables_X=variables_X
        )
        for _ in range(numero_particulas)
    ]

    velocidades = [
        crear_velocidad_Z_inicial(generador, particula)
        for particula in enjambre
    ]

    resultados_actuales = [
        evaluar_particula_Z(
            particula=particula,
            dataframe=dataframe,
            objetivo=objetivo,
            activacion_X=activacion_X,
            variables_X=variables_X,
            valor_consecuente_dominante=valor_consecuente_dominante,
            metodo_similitud=metodo_similitud,
            soporte_minimo=soporte_minimo
        )
        for particula in enjambre
    ]

    pbest_particulas = copy.deepcopy(enjambre)
    pbest_resultados = copy.deepcopy(resultados_actuales)
    pbest_fitness = np.array([
        resultado["fitness"] for resultado in resultados_actuales
    ])

    indice_gbest = int(np.argmax(pbest_fitness))
    gbest_particula = copy.deepcopy(pbest_particulas[indice_gbest])
    gbest_fitness = float(pbest_fitness[indice_gbest])
    gbest_resultado = copy.deepcopy(pbest_resultados[indice_gbest])

    metricas_gbest = gbest_resultado["metricas"]
    fitness_actuales = np.array([
        resultado["fitness"] for resultado in resultados_actuales
    ])

    historial = [{
        "iteracion": 0,
        "gbest_fitness": gbest_fitness,
        "fitness_promedio_actual": float(fitness_actuales.mean()),
        "fitness_maximo_actual": float(fitness_actuales.max()),
        "particulas_fitness_positivo": int(
            np.count_nonzero(fitness_actuales > 0)
        ),
        "pbest_actualizados": 0,
        "numero_condiciones_gbest": len(
            gbest_resultado["condiciones_Z"]
        ),
        "soporte_anomalo_gbest": metricas_gbest["soporte_anomalo"],
        "confianza_anomala_gbest": metricas_gbest["confianza_anomala"],
        "cf_directo_gbest": metricas_gbest["cf_Z_hacia_noY"],
        "cf_inverso_gbest": metricas_gbest["cf_noY_hacia_Z"]
    }]

    for iteracion in range(1, numero_iteraciones + 1):
        resultado_iteracion = ejecutar_iteracion_pso_anomalo(
            enjambre=enjambre,
            velocidades=velocidades,
            pbest_particulas=pbest_particulas,
            pbest_fitness=pbest_fitness,
            pbest_resultados=pbest_resultados,
            gbest_particula=gbest_particula,
            gbest_fitness=gbest_fitness,
            gbest_resultado=gbest_resultado,
            dataframe=dataframe,
            objetivo=objetivo,
            activacion_X=activacion_X,
            variables_X=variables_X,
            valor_consecuente_dominante=valor_consecuente_dominante,
            metodo_similitud=metodo_similitud,
            soporte_minimo=soporte_minimo,
            generador=generador,
            probabilidad_exploracion=probabilidad_exploracion
        )

        enjambre = resultado_iteracion["enjambre"]
        velocidades = resultado_iteracion["velocidades"]
        resultados_actuales = resultado_iteracion["resultados"]
        pbest_particulas = resultado_iteracion["pbest_particulas"]
        pbest_fitness = resultado_iteracion["pbest_fitness"]
        pbest_resultados = resultado_iteracion["pbest_resultados"]
        gbest_particula = resultado_iteracion["gbest_particula"]
        gbest_fitness = resultado_iteracion["gbest_fitness"]
        gbest_resultado = resultado_iteracion["gbest_resultado"]

        historial.append({
            "iteracion": iteracion,
            **resultado_iteracion["resumen"]
        })

    return {
        "pbest_particulas": pbest_particulas,
        "pbest_fitness": pbest_fitness,
        "pbest_resultados": pbest_resultados,
        "gbest_particula": gbest_particula,
        "gbest_fitness": gbest_fitness,
        "gbest_resultado": gbest_resultado,
        "historial_convergencia": pd.DataFrame(historial),
        "configuracion": {
            "numero_particulas": numero_particulas,
            "numero_iteraciones": numero_iteraciones,
            "soporte_minimo": soporte_minimo,
            "semilla": semilla,
            "metodo_similitud": metodo_similitud,
            "valor_consecuente_dominante": valor_consecuente_dominante,
            "probabilidad_exploracion": probabilidad_exploracion
        }
    }

In [ ]:
# ============================================================
# PREPARAR FUNCIONES PARA LA VALIDACIÓN FINAL
# ============================================================

def preparar_contexto_dominante(regla_dominante):
    """Reconstruye el antecedente X de una regla dominante."""
    resultado = regla_dominante["resultado"]
    metricas = resultado["metricas"]
    condiciones_X = resultado["condiciones"]

    valor_consecuente = int(
        regla_dominante.get(
            "valor_consecuente",
            metricas["valor_consecuente"]
        )
    )

    metodo_similitud = regla_dominante.get(
        "metodo_similitud",
        metricas["metodo_similitud"]
    )

    variables_X = [
        condicion["variable"]
        for condicion in condiciones_X
    ]

    activacion_X = calcular_activacion_antecedente(
        X_datos,
        condiciones_X,
        metodo_similitud
    )

    return {
        "resultado": resultado,
        "condiciones_X": condiciones_X,
        "variables_X": variables_X,
        "activacion_X": activacion_X,
        "valor_consecuente": valor_consecuente,
        "valor_anomalo": 1 - valor_consecuente,
        "metodo_similitud": metodo_similitud
    }


def describir_condiciones_Z(condiciones_Z):
    """Convierte Z en una descripción compacta."""
    partes = []

    for condicion in condiciones_Z:
        variable = condicion["variable"]

        if condicion["tipo"] == "continua":
            partes.append(
                f"{variable}(c={condicion['centro']:.3f}, "
                f"r={condicion['radio']:.3f})"
            )
        else:
            partes.append(
                f"{variable}={condicion['valor']}"
            )

    return " Y ".join(partes)


print("Funciones auxiliares preparadas.")

## 10. Comparacion exploratoria opcional

La comparacion de 50, 100 y 500 particulas usa una sola regla dominante y una sola semilla. Sirve como diagnostico de costo y comportamiento, no como el experimento anomaloso canonico. Por defecto esta desactivada y no modifica las variables del flujo principal.


In [ ]:
# Configuración de la comparación inicial
PARTICULAS_COMPARACION = [50, 100, 500]
ITERACIONES_COMPARACION = 50
SOPORTE_COMPARACION = 0.001
SEMILLA_COMPARACION = 55

In [ ]:
EJECUTAR_COMPARACION_INICIAL = False

if EJECUTAR_COMPARACION_INICIAL:
    # ============================================================
    # COMPARAR TAMAÑOS DE ENJAMBRE
    # ============================================================

    resultados_comparacion_particulas = {}
    filas_comparacion_particulas = []

    for numero_particulas in PARTICULAS_COMPARACION:
        print(f"Ejecutando PSO con {numero_particulas} partículas...")
        inicio = pd.Timestamp.now()

        resultado_pso = ejecutar_pso_anomalo(
            dataframe=X_datos,
            objetivo=y_objetivo,
            activacion_X=activacion_X,
            variables_X=variables_X,
            valor_consecuente_dominante=valor_consecuente,
            metodo_similitud=metodo_similitud,
            soporte_minimo=SOPORTE_COMPARACION,
            numero_particulas=numero_particulas,
            numero_iteraciones=ITERACIONES_COMPARACION,
            semilla=SEMILLA_COMPARACION
        )

        tiempo_segundos = (
            pd.Timestamp.now() - inicio
        ).total_seconds()

        resultados_comparacion_particulas[numero_particulas] = resultado_pso
        metricas_gbest = resultado_pso["gbest_resultado"]["metricas"]

        fila = {
            "particulas": numero_particulas,
            "iteraciones": ITERACIONES_COMPARACION,
            "tiempo_segundos": tiempo_segundos,
            "gbest_fitness": resultado_pso["gbest_fitness"],
            "longitud_Z_gbest": len(
                resultado_pso["gbest_resultado"]["condiciones_Z"]
            ),
            "soporte_anomalo_gbest": metricas_gbest["soporte_anomalo"],
            "confianza_anomala_gbest": metricas_gbest["confianza_anomala"],
            "confianza_inversa_gbest": metricas_gbest["confianza_inversa"],
            "cf_directo_gbest": metricas_gbest["cf_Z_hacia_noY"],
            "cf_inverso_gbest": metricas_gbest["cf_noY_hacia_Z"],
            "pbest_fitness_positivo": int(
                np.count_nonzero(resultado_pso["pbest_fitness"] > 0)
            )
        }

        for confianza_minima in CONFIANZAS_ANOMALAS:
            etiqueta = int(confianza_minima * 100)

            validaciones_pbest = [
                verificar_regla_anomala(
                    resultado_regla=resultado_regla,
                    soporte_minimo=SOPORTE_COMPARACION,
                    confianza_minima=confianza_minima,
                    min_condiciones=MIN_CONDICIONES_Z,
                    max_condiciones=MAX_CONDICIONES_Z
                )["es_anomala"]
                for resultado_regla in resultado_pso["pbest_resultados"]
            ]

            validacion_gbest = verificar_regla_anomala(
                resultado_regla=resultado_pso["gbest_resultado"],
                soporte_minimo=SOPORTE_COMPARACION,
                confianza_minima=confianza_minima,
                min_condiciones=MIN_CONDICIONES_Z,
                max_condiciones=MAX_CONDICIONES_Z
            )

            fila[f"pbest_aceptados_conf_{etiqueta}"] = int(
                np.sum(validaciones_pbest)
            )
            fila[f"gbest_aceptado_conf_{etiqueta}"] = (
                validacion_gbest["es_anomala"]
            )

        filas_comparacion_particulas.append(fila)

        print(
            f"Finalizado en {tiempo_segundos:.1f} segundos. "
            f"Fitness gBest: {resultado_pso['gbest_fitness']:.6f}"
        )

    tabla_comparacion_particulas = pd.DataFrame(
        filas_comparacion_particulas
    )

    display(
        tabla_comparacion_particulas.round({
            "tiempo_segundos": 1,
            "gbest_fitness": 8,
            "soporte_anomalo_gbest": 6,
            "confianza_anomala_gbest": 6,
            "confianza_inversa_gbest": 6,
            "cf_directo_gbest": 6,
            "cf_inverso_gbest": 6
        })
    )
else:
    resultados_comparacion_particulas = {}
    tabla_comparacion_particulas = pd.DataFrame()
    print("Comparacion exploratoria omitida; continua el flujo canonico.")


In [ ]:
# ============================================================
# ANALIZAR FACTIBILIDAD DE LAS REGLAS DOMINANTES
# ============================================================

filas_factibilidad = []

for indice, regla_dominante in enumerate(
    reglas_dominantes_principales
):
    contexto = preparar_contexto_dominante(regla_dominante)
    activacion_X_regla = contexto["activacion_X"]

    activacion_noY = (
        y_objetivo.to_numpy() == contexto["valor_anomalo"]
    ).astype(float)

    activacion_X_noY = np.minimum(
        activacion_X_regla,
        activacion_noY
    )

    masa_X = activacion_X_regla.sum()
    masa_X_noY = activacion_X_noY.sum()
    soporte_maximo_anomalo = masa_X_noY / len(y_objetivo)

    fila_catalogo = catalogo_dominantes_principal.iloc[indice]

    fila = {
        "indice": indice,
        "regla_id": fila_catalogo.get(
            "regla_id",
            f"RD_{indice + 1:04d}"
        ),
        "regla": fila_catalogo["regla"],
        "metodo_similitud": contexto["metodo_similitud"],
        "consecuente_dominante": contexto["valor_consecuente"],
        "numero_condiciones_X": len(contexto["condiciones_X"]),
        "variables_disponibles_Z": (
            len(columnas_antecedente) - len(contexto["variables_X"])
        ),
        "masa_X": masa_X,
        "masa_fallos_X": masa_X_noY,
        "proporcion_fallos_en_X": (
            masa_X_noY / masa_X if masa_X > 0 else 0.0
        ),
        "soporte_maximo_anomalo": soporte_maximo_anomalo
    }

    for soporte_minimo in SOPORTES_ANOMALOS:
        etiqueta = str(soporte_minimo).replace(".", "_")
        fila[f"factible_supp_{etiqueta}"] = (
            soporte_maximo_anomalo >= soporte_minimo
        )

    filas_factibilidad.append(fila)

tabla_factibilidad_dominantes = pd.DataFrame(
    filas_factibilidad
)

resumen_factibilidad = pd.DataFrame([
    {
        "min_supp": soporte_minimo,
        "masa_minima": soporte_minimo * len(y_objetivo),
        "reglas_factibles": int(
            tabla_factibilidad_dominantes[
                f"factible_supp_{str(soporte_minimo).replace('.', '_')}"
            ].sum()
        ),
        "reglas_no_factibles": int(
            (
                ~tabla_factibilidad_dominantes[
                    f"factible_supp_{str(soporte_minimo).replace('.', '_')}"
                ]
            ).sum()
        )
    }
    for soporte_minimo in SOPORTES_ANOMALOS
])

print("Factibilidad de la regla utilizada en la prueba:")

display(
    tabla_factibilidad_dominantes.head(1).round({
        "masa_X": 2,
        "masa_fallos_X": 2,
        "proporcion_fallos_en_X": 6,
        "soporte_maximo_anomalo": 6
    })
)

print("\nResumen de las 166 reglas dominantes:")

display(resumen_factibilidad.round(4))

## 12. Experimento anomaloso canonico

Esta seccion ejecuta el barrido completo sobre las reglas dominantes factibles. La configuracion documentada es:

$$
N_p=35,\qquad T=40,\qquad s=55
$$

Se evaluan los soportes minimos $\{0.0005,0.001,0.002\}$ y se aplican posteriormente las confianzas minimas $\{0.60,0.70,0.80\}$. Las nueve combinaciones de umbrales se resumen a partir de las mismas ejecuciones PSO; no se repite innecesariamente una busqueda por cada confianza.

Cada candidata conserva la estructura $R_a:X\land Z\Rightarrow\neg Y$, su regla dominante de referencia, sus metricas y su activacion para la consolidacion posterior.


In [ ]:
# ============================================================
# EJECUTAR EXPERIMENTO CON 35 PARTÍCULAS
# ============================================================

PARTICULAS_EXPERIMENTO = 35
ITERACIONES_EXPERIMENTO = 40
SEMILLA_EXPERIMENTO = 55
GUARDAR_CADA_REGLAS = 5

carpeta_experimento = (
    f"{carpeta_base}/experimento_anomalias_35_particulas"
)

os.makedirs(carpeta_experimento, exist_ok=True)

ruta_checkpoint = (
    f"{carpeta_experimento}/checkpoint_experimento.pkl"
)

ruta_resultados = (
    f"{carpeta_experimento}/resultados_experimento.pkl"
)

ruta_resumen_ejecuciones = (
    f"{carpeta_experimento}/resumen_ejecuciones.csv"
)

ruta_resumen_configuraciones = (
    f"{carpeta_experimento}/resumen_configuraciones.csv"
)


def ejecutar_experimento_anomalias():
    """Ejecuta las reglas factibles y permite reanudar el proceso."""
    configuracion = {
        "particulas": PARTICULAS_EXPERIMENTO,
        "iteraciones": ITERACIONES_EXPERIMENTO,
        "semilla": SEMILLA_EXPERIMENTO,
        "soportes": SOPORTES_ANOMALOS,
        "confianzas": CONFIANZAS_ANOMALAS,
        "reglas_dominantes": len(reglas_dominantes_principales)
    }

    if os.path.exists(ruta_checkpoint):
        estado = joblib.load(ruta_checkpoint)

        if estado["configuracion"] != configuracion:
            raise ValueError(
                "El checkpoint pertenece a otra configuración."
            )

        indice_inicial = estado["ultimo_indice"] + 1
        print(f"Reanudando desde la regla {indice_inicial + 1}.")
    else:
        estado = {
            "configuracion": configuracion,
            "ultimo_indice": -1,
            "resumen_ejecuciones": [],
            "candidatas": []
        }
        indice_inicial = 0

    total_reglas = len(reglas_dominantes_principales)

    for indice in range(indice_inicial, total_reglas):
        regla_dominante = reglas_dominantes_principales[indice]
        fila_catalogo = catalogo_dominantes_principal.iloc[indice]
        fila_factibilidad = tabla_factibilidad_dominantes.iloc[indice]

        regla_id = fila_catalogo.get(
            "regla_id",
            f"RD_{indice + 1:04d}"
        )

        contexto = preparar_contexto_dominante(
            regla_dominante
        )

        for soporte_minimo in SOPORTES_ANOMALOS:
            etiqueta_soporte = str(
                soporte_minimo
            ).replace(".", "_")

            es_factible = bool(
                fila_factibilidad[
                    f"factible_supp_{etiqueta_soporte}"
                ]
            )

            if not es_factible:
                fila_resumen = {
                    "indice_dominante": indice,
                    "regla_id_dominante": regla_id,
                    "metodo_similitud": contexto[
                        "metodo_similitud"
                    ],
                    "consecuente_dominante": contexto[
                        "valor_consecuente"
                    ],
                    "soporte_minimo": soporte_minimo,
                    "estado": "no_factible",
                    "tiempo_segundos": 0.0
                }

                for confianza in CONFIANZAS_ANOMALAS:
                    etiqueta = int(confianza * 100)
                    fila_resumen[
                        f"pbest_aceptados_conf_{etiqueta}"
                    ] = 0

                estado["resumen_ejecuciones"].append(
                    fila_resumen
                )
                continue

            inicio = pd.Timestamp.now()

            resultado_pso = ejecutar_pso_anomalo(
                dataframe=X_datos,
                objetivo=y_objetivo,
                activacion_X=contexto["activacion_X"],
                variables_X=contexto["variables_X"],
                valor_consecuente_dominante=contexto[
                    "valor_consecuente"
                ],
                metodo_similitud=contexto[
                    "metodo_similitud"
                ],
                soporte_minimo=soporte_minimo,
                numero_particulas=PARTICULAS_EXPERIMENTO,
                numero_iteraciones=ITERACIONES_EXPERIMENTO,
                semilla=SEMILLA_EXPERIMENTO
            )

            tiempo_segundos = (
                pd.Timestamp.now() - inicio
            ).total_seconds()

            conteos_confianza = {
                confianza: 0
                for confianza in CONFIANZAS_ANOMALAS
            }

            for indice_particula, resultado_regla in enumerate(
                resultado_pso["pbest_resultados"],
                start=1
            ):
                confianzas_superadas = []

                for confianza_minima in CONFIANZAS_ANOMALAS:
                    validacion = verificar_regla_anomala(
                        resultado_regla=resultado_regla,
                        soporte_minimo=soporte_minimo,
                        confianza_minima=confianza_minima,
                        min_condiciones=MIN_CONDICIONES_Z,
                        max_condiciones=MAX_CONDICIONES_Z
                    )

                    if validacion["es_anomala"]:
                        conteos_confianza[
                            confianza_minima
                        ] += 1

                        confianzas_superadas.append(
                            confianza_minima
                        )

                if confianzas_superadas:
                    estado["candidatas"].append({
                        "indice_dominante": indice,
                        "regla_id_dominante": regla_id,
                        "soporte_busqueda": soporte_minimo,
                        "semilla": SEMILLA_EXPERIMENTO,
                        "particula": indice_particula,
                        "metodo_similitud": contexto[
                            "metodo_similitud"
                        ],
                        "consecuente_dominante": contexto[
                            "valor_consecuente"
                        ],
                        "consecuente_anomalo": contexto[
                            "valor_anomalo"
                        ],
                        "confianzas_superadas": tuple(
                            confianzas_superadas
                        ),
                        "resultado": copy.deepcopy(
                            resultado_regla
                        )
                    })

            metricas_gbest = resultado_pso[
                "gbest_resultado"
            ]["metricas"]

            fila_resumen = {
                "indice_dominante": indice,
                "regla_id_dominante": regla_id,
                "metodo_similitud": contexto[
                    "metodo_similitud"
                ],
                "consecuente_dominante": contexto[
                    "valor_consecuente"
                ],
                "soporte_minimo": soporte_minimo,
                "estado": "ejecutado",
                "tiempo_segundos": tiempo_segundos,
                "gbest_fitness": resultado_pso[
                    "gbest_fitness"
                ],
                "soporte_anomalo_gbest": metricas_gbest[
                    "soporte_anomalo"
                ],
                "confianza_anomala_gbest": metricas_gbest[
                    "confianza_anomala"
                ],
                "confianza_inversa_gbest": metricas_gbest[
                    "confianza_inversa"
                ],
                "cf_directo_gbest": metricas_gbest[
                    "cf_Z_hacia_noY"
                ],
                "cf_inverso_gbest": metricas_gbest[
                    "cf_noY_hacia_Z"
                ],
                "numero_condiciones_gbest": len(
                    resultado_pso[
                        "gbest_resultado"
                    ]["condiciones_Z"]
                ),
                "pbest_fitness_positivo": int(
                    np.count_nonzero(
                        resultado_pso["pbest_fitness"] > 0
                    )
                )
            }

            for confianza, cantidad in conteos_confianza.items():
                etiqueta = int(confianza * 100)
                fila_resumen[
                    f"pbest_aceptados_conf_{etiqueta}"
                ] = cantidad

            estado["resumen_ejecuciones"].append(
                fila_resumen
            )

            del resultado_pso
            gc.collect()

        estado["ultimo_indice"] = indice

        if (
            (indice + 1) % GUARDAR_CADA_REGLAS == 0
            or indice + 1 == total_reglas
        ):
            joblib.dump(estado, ruta_checkpoint)

            print(
                f"Procesadas {indice + 1}/{total_reglas} reglas. "
                f"Candidatas: {len(estado['candidatas'])}"
            )

    return estado


estado_experimento = ejecutar_experimento_anomalias()

tabla_resumen_ejecuciones = pd.DataFrame(
    estado_experimento["resumen_ejecuciones"]
)

candidatas_experimento = estado_experimento[
    "candidatas"
]

filas_resumen_configuraciones = []

for soporte_minimo, confianza_minima in product(
    SOPORTES_ANOMALOS,
    CONFIANZAS_ANOMALAS
):
    candidatas_configuracion = [
        candidata
        for candidata in candidatas_experimento
        if candidata["soporte_busqueda"] == soporte_minimo
        and confianza_minima
        in candidata["confianzas_superadas"]
    ]

    reglas_con_anomalia = {
        candidata["regla_id_dominante"]
        for candidata in candidatas_configuracion
    }

    filas_resumen_configuraciones.append({
        "min_supp": soporte_minimo,
        "min_conf": confianza_minima,
        "candidatas_pbest": len(
            candidatas_configuracion
        ),
        "reglas_dominantes_con_anomalia": len(
            reglas_con_anomalia
        ),
        "reglas_dominantes_sin_anomalia": (
            len(reglas_dominantes_principales)
            - len(reglas_con_anomalia)
        )
    })

tabla_resumen_configuraciones = pd.DataFrame(
    filas_resumen_configuraciones
)

resultados_experimento = {
    "configuracion": estado_experimento[
        "configuracion"
    ],
    "resumen_ejecuciones": tabla_resumen_ejecuciones,
    "candidatas": candidatas_experimento,
    "resumen_configuraciones": (
        tabla_resumen_configuraciones
    )
}

joblib.dump(
    resultados_experimento,
    ruta_resultados
)

tabla_resumen_ejecuciones.to_csv(
    ruta_resumen_ejecuciones,
    index=False
)

tabla_resumen_configuraciones.to_csv(
    ruta_resumen_configuraciones,
    index=False
)

print(
    f"Experimento terminado.\n"
    f"Ejecuciones registradas: "
    f"{len(tabla_resumen_ejecuciones)}\n"
    f"Candidatas pBest: "
    f"{len(candidatas_experimento)}\n"
    f"Resultados: {ruta_resultados}"
)

display(tabla_resumen_configuraciones)

In [ ]:
# ============================================================
# ORGANIZAR CANDIDATAS ANOMALAS
# ============================================================

filas_candidatas = []

for candidata in candidatas_experimento:
    resultado = candidata["resultado"]
    metricas = resultado["metricas"]
    condiciones_Z = resultado["condiciones_Z"]

    confianzas_superadas = candidata[
        "confianzas_superadas"
    ]

    filas_candidatas.append({
        "regla_id_dominante": candidata[
            "regla_id_dominante"
        ],
        "soporte_busqueda": candidata[
            "soporte_busqueda"
        ],
        "confianzas_superadas": ", ".join(
            f"{conf:.2f}"
            for conf in confianzas_superadas
        ),
        "mejor_confianza_minima": max(
            confianzas_superadas
        ),
        "particula": candidata["particula"],
        "metodo_similitud": candidata[
            "metodo_similitud"
        ],
        "consecuente_dominante": candidata[
            "consecuente_dominante"
        ],
        "consecuente_anomalo": candidata[
            "consecuente_anomalo"
        ],
        "Z": describir_condiciones_Z(
            condiciones_Z
        ),
        "numero_condiciones_Z": len(
            condiciones_Z
        ),
        "fitness": resultado["fitness"],
        "soporte_anomalo": metricas[
            "soporte_anomalo"
        ],
        "confianza_anomala": metricas[
            "confianza_anomala"
        ],
        "confianza_inversa": metricas[
            "confianza_inversa"
        ],
        "cf_directo": metricas[
            "cf_Z_hacia_noY"
        ],
        "cf_inverso": metricas[
            "cf_noY_hacia_Z"
        ]
    })

tabla_candidatas_anomalas = pd.DataFrame(
    filas_candidatas
)

tabla_candidatas_anomalas = (
    tabla_candidatas_anomalas
    .sort_values(
        by=[
            "mejor_confianza_minima",
            "fitness"
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

ruta_candidatas = (
    f"{carpeta_experimento}/"
    "candidatas_anomalas_sin_filtrar.csv"
)

tabla_candidatas_anomalas.to_csv(
    ruta_candidatas,
    index=False
)

print(
    f"Candidatas registradas: "
    f"{len(tabla_candidatas_anomalas)}"
)

print("\nCandidatas que superan confianza mínima 0.70:")

display(
    tabla_candidatas_anomalas[
        tabla_candidatas_anomalas[
            "mejor_confianza_minima"
        ] >= 0.70
    ].head(5).round({
        "fitness": 8,
        "soporte_anomalo": 6,
        "confianza_anomala": 6,
        "confianza_inversa": 6,
        "cf_directo": 6,
        "cf_inverso": 6
    })
)

print("\nCantidad de candidatas por regla dominante:")

display(
    tabla_candidatas_anomalas
    .groupby("regla_id_dominante")
    .agg(
        candidatas=("regla_id_dominante", "count"),
        mejor_fitness=("fitness", "max"),
        mejor_confianza=("confianza_anomala", "max"),
        mejor_cf_directo=("cf_directo", "max")
    )
    .sort_values("mejor_fitness", ascending=False)
    .reset_index()
    .head(5
          )
)

In [ ]:
# ============================================================
# ELIMINAR REDUNDANCIA CON JACCARD DIFUSO
# ============================================================

UMBRAL_JACCARD_ANOMALIAS = 0.85


def jaccard_difuso(activacion_a, activacion_b):
    """Calcula el Jaccard difuso entre dos activaciones."""
    activacion_a = np.asarray(activacion_a, dtype=float)
    activacion_b = np.asarray(activacion_b, dtype=float)

    interseccion = np.minimum(
        activacion_a,
        activacion_b
    ).sum()

    union = np.maximum(
        activacion_a,
        activacion_b
    ).sum()

    return float(interseccion / union) if union > 0 else 0.0


# Priorizar primero mayor confianza y después mayor fitness
candidatas_ordenadas = sorted(
    candidatas_experimento,
    key=lambda candidata: (
        max(candidata["confianzas_superadas"]),
        candidata["resultado"]["fitness"]
    ),
    reverse=True
)

representantes_por_regla = {}
activaciones_por_regla = {}
candidatas_descartadas = []
numero_comparaciones_jaccard = 0

for candidata in candidatas_ordenadas:
    indice_dominante = candidata["indice_dominante"]
    regla_id = candidata["regla_id_dominante"]

    if regla_id not in representantes_por_regla:
        contexto = preparar_contexto_dominante(
            reglas_dominantes_principales[indice_dominante]
        )

        representantes_por_regla[regla_id] = []
        activaciones_por_regla[regla_id] = {
            "contexto": contexto,
            "activaciones": []
        }

    contexto = activaciones_por_regla[regla_id]["contexto"]
    condiciones_Z = candidata["resultado"]["condiciones_Z"]

    activacion_Z = calcular_activacion_antecedente(
        X_datos,
        condiciones_Z,
        candidata["metodo_similitud"]
    )

    activacion_XZ = np.minimum(
        contexto["activacion_X"],
        activacion_Z
    ).astype(np.float32)

    representantes = representantes_por_regla[regla_id]
    activaciones_representantes = (
        activaciones_por_regla[regla_id]["activaciones"]
    )

    es_redundante = False
    mayor_jaccard = 0.0
    indice_representante = None

    for indice_representante_actual, activacion_representante in enumerate(
        activaciones_representantes
    ):
        numero_comparaciones_jaccard += 1

        similitud = jaccard_difuso(
            activacion_XZ,
            activacion_representante
        )

        if similitud > mayor_jaccard:
            mayor_jaccard = similitud
            indice_representante = indice_representante_actual

        if similitud >= UMBRAL_JACCARD_ANOMALIAS:
            es_redundante = True
            break

    if es_redundante:
        representante = representantes[indice_representante]

        representante["miembros_familia_jaccard"] += 1
        representante["jaccard_maximo_familia"] = max(
            representante["jaccard_maximo_familia"],
            mayor_jaccard
        )

        representante["confianzas_familia"].update(
            candidata["confianzas_superadas"]
        )

        representante["soportes_familia"].add(
            candidata["soporte_busqueda"]
        )

        candidatas_descartadas.append({
            "regla_id_dominante": regla_id,
            "soporte_busqueda": candidata[
                "soporte_busqueda"
            ],
            "fitness_descartado": candidata[
                "resultado"
            ]["fitness"],
            "jaccard_representante": mayor_jaccard,
            "fitness_representante": representante[
                "resultado"
            ]["fitness"]
        })

    else:
        nuevo_representante = copy.deepcopy(candidata)

        nuevo_representante[
            "miembros_familia_jaccard"
        ] = 1

        nuevo_representante[
            "jaccard_maximo_familia"
        ] = 0.0

        nuevo_representante[
            "confianzas_familia"
        ] = set(
            candidata["confianzas_superadas"]
        )

        nuevo_representante[
            "soportes_familia"
        ] = {
            candidata["soporte_busqueda"]
        }

        representantes.append(
            nuevo_representante
        )

        activaciones_representantes.append(
            activacion_XZ
        )


anomalias_no_redundantes = [
    candidata
    for representantes in representantes_por_regla.values()
    for candidata in representantes
]

anomalias_no_redundantes.sort(
    key=lambda candidata: (
        max(candidata["confianzas_familia"]),
        candidata["resultado"]["fitness"]
    ),
    reverse=True
)


# Crear tabla interpretada
filas_anomalias = []

for posicion, candidata in enumerate(
    anomalias_no_redundantes,
    start=1
):
    resultado = candidata["resultado"]
    metricas = resultado["metricas"]
    indice_dominante = candidata["indice_dominante"]

    filas_anomalias.append({
        "posicion": posicion,
        "regla_id_dominante": candidata[
            "regla_id_dominante"
        ],
        "regla_dominante": catalogo_dominantes_principal.iloc[
            indice_dominante
        ]["regla"],
        "soporte_busqueda": candidata[
            "soporte_busqueda"
        ],
        "confianzas_familia": ", ".join(
            f"{valor:.2f}"
            for valor in sorted(
                candidata["confianzas_familia"],
                reverse=True
            )
        ),
        "metodo_similitud": candidata[
            "metodo_similitud"
        ],
        "Z": describir_condiciones_Z(
            resultado["condiciones_Z"]
        ),
        "numero_condiciones_Z": len(
            resultado["condiciones_Z"]
        ),
        "fitness": resultado["fitness"],
        "soporte_anomalo": metricas[
            "soporte_anomalo"
        ],
        "confianza_anomala": metricas[
            "confianza_anomala"
        ],
        "confianza_inversa": metricas[
            "confianza_inversa"
        ],
        "cf_directo": metricas[
            "cf_Z_hacia_noY"
        ],
        "cf_inverso": metricas[
            "cf_noY_hacia_Z"
        ],
        "miembros_familia_jaccard": candidata[
            "miembros_familia_jaccard"
        ],
        "jaccard_maximo_familia": candidata[
            "jaccard_maximo_familia"
        ]
    })


tabla_anomalias_no_redundantes = pd.DataFrame(
    filas_anomalias
)

resultado_redundancia_anomalias = {
    "reglas_no_redundantes": anomalias_no_redundantes,
    "tabla_reglas": tabla_anomalias_no_redundantes,
    "candidatas_descartadas": candidatas_descartadas,
    "umbral_jaccard": UMBRAL_JACCARD_ANOMALIAS,
    "numero_comparaciones": numero_comparaciones_jaccard
}

ruta_anomalias_consolidadas = (
    f"{carpeta_experimento}/"
    "anomalias_no_redundantes.pkl"
)

ruta_tabla_anomalias = (
    f"{carpeta_experimento}/"
    "anomalias_no_redundantes.csv"
)

joblib.dump(
    resultado_redundancia_anomalias,
    ruta_anomalias_consolidadas
)

tabla_anomalias_no_redundantes.to_csv(
    ruta_tabla_anomalias,
    index=False
)

print(
    f"Candidatas iniciales: "
    f"{len(candidatas_experimento)}\n"
    f"Anomalías no redundantes: "
    f"{len(anomalias_no_redundantes)}\n"
    f"Candidatas agrupadas como redundantes: "
    f"{len(candidatas_descartadas)}\n"
    f"Comparaciones Jaccard: "
    f"{numero_comparaciones_jaccard}"
)

display(
    tabla_anomalias_no_redundantes.head(30).round({
        "fitness": 8,
        "soporte_anomalo": 6,
        "confianza_anomala": 6,
        "confianza_inversa": 6,
        "cf_directo": 6,
        "cf_inverso": 6,
        "jaccard_maximo_familia": 6
    })
)

In [ ]:
# ============================================================
# RESUMIR Y SELECCIONAR CANDIDATAS ANÓMALAS
# ============================================================

tabla_candidatas_anomalas = tabla_anomalias_no_redundantes.copy()

if tabla_candidatas_anomalas.empty:
    raise ValueError("No se encontraron candidatas anómalas.")

tabla_candidatas_anomalas["nivel_confianza"] = np.select(
    [
        tabla_candidatas_anomalas["confianza_anomala"] >= 0.70,
        tabla_candidatas_anomalas["confianza_anomala"] >= 0.60
    ],
    [
        "alta",
        "aceptable"
    ],
    default="inferior"
)

tabla_candidatas_prioritarias = (
    tabla_candidatas_anomalas[
        tabla_candidatas_anomalas["confianza_anomala"] >= 0.70
    ]
    .sort_values(
        ["confianza_anomala", "fitness", "soporte_anomalo"],
        ascending=False
    )
    .reset_index(drop=True)
)

resumen_reglas_padre = (
    tabla_candidatas_anomalas
    .groupby("regla_id_dominante")
    .agg(
        candidatas=("regla_id_dominante", "size"),
        mejor_confianza=("confianza_anomala", "max"),
        mejor_soporte=("soporte_anomalo", "max"),
        mejor_fitness=("fitness", "max"),
        maximo_Z=("numero_condiciones_Z", "max"),
        metodos=(
            "metodo_similitud",
            lambda valores: ", ".join(sorted(set(valores)))
        )
    )
    .sort_values(
        ["mejor_confianza", "mejor_fitness"],
        ascending=False
    )
    .reset_index()
)

print(f"Candidatas no redundantes: {len(tabla_candidatas_anomalas)}")
print(
    f"Candidatas con confianza >= 0.70: "
    f"{len(tabla_candidatas_prioritarias)}"
)
print(
    f"Reglas dominantes con alguna candidata: "
    f"{tabla_anomalias_no_redundantes['regla_id_dominante'].nunique()}"
)

print("\nCandidatas prioritarias:")
display(
    tabla_candidatas_prioritarias[
        [
            "regla_id_dominante",
            "regla_dominante",
            "Z",
            "numero_condiciones_Z",
            "soporte_anomalo",
            "confianza_anomala",
            "confianza_inversa",
            "fitness",
            "miembros_familia_jaccard"
        ]
    ].round(6)
)

print("\nResumen por regla dominante:")
display(resumen_reglas_padre.round(6))

ruta_prioritarias = (
    f"{carpeta_experimento}/"
    "candidatas_anomalas_prioritarias.csv"
)

tabla_candidatas_prioritarias.to_csv(
    ruta_prioritarias,
    index=False
)

print(f"\nArchivo guardado: {ruta_prioritarias}")

In [ ]:
# ============================================================
# PREPARAR VALIDACIÓN DE ESTABILIDAD
# ============================================================

ids_alta_confianza = (
    tabla_candidatas_anomalas
    .loc[
        tabla_candidatas_anomalas["confianza_anomala"] >= 0.70,
        "regla_id_dominante"
    ]
    .drop_duplicates()
    .tolist()
)

id_mejor_fitness = (
    tabla_candidatas_anomalas
    .sort_values("fitness", ascending=False)
    .iloc[0]["regla_id_dominante"]
)

ids_validacion = list(dict.fromkeys(
    ids_alta_confianza + [id_mejor_fitness]
))

indices_por_id = dict(zip(
    catalogo_dominantes_principal["regla_id"],
    range(len(catalogo_dominantes_principal))
))

reglas_dominantes_validacion = [
    reglas_dominantes_principales[indices_por_id[regla_id]]
    for regla_id in ids_validacion
]

tabla_reglas_validacion = (
    tabla_candidatas_anomalas[
        tabla_candidatas_anomalas[
            "regla_id_dominante"
        ].isin(ids_validacion)
    ]
    .sort_values(
        ["regla_id_dominante", "confianza_anomala"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

SEMILLAS_VALIDACION = [101, 102, 103]
PARTICULAS_VALIDACION = 100
ITERACIONES_VALIDACION = 50
SOPORTES_VALIDACION = [0.0005, 0.001, 0.002]
CONFIANZAS_VALIDACION = [0.60, 0.70]

print(f"Reglas seleccionadas: {ids_validacion}")
print(f"Ejecuciones previstas: {len(ids_validacion) * len(SEMILLAS_VALIDACION) * len(SOPORTES_VALIDACION) * len(CONFIANZAS_VALIDACION)}")

display(
    tabla_reglas_validacion[
        [
            "regla_id_dominante",
            "Z",
            "soporte_anomalo",
            "confianza_anomala",
            "confianza_inversa",
            "fitness"
        ]
    ].round(6)
)

In [ ]:
# ============================================================
# GUARDAR ESTADO DEL EXPERIMENTO ACTUAL
# ============================================================

semilla_actual = globals().get(
    "SEMILLA_EXPERIMENTO",
    55
)

particulas_actuales = globals().get(
    "PARTICULAS_EXPERIMENTO",
    35
)

iteraciones_actuales = globals().get(
    "ITERACIONES_EXPERIMENTO",
    40
)

estado_experimento_actual = {
    "configuracion": {
        "particulas": particulas_actuales,
        "iteraciones": iteraciones_actuales,
        "semilla": semilla_actual
    },
    "tabla_anomalias_no_redundantes": (
        tabla_anomalias_no_redundantes
    ),
    "resultado_redundancia_anomalias": (
        resultado_redundancia_anomalias
    ),
    "tabla_candidatas_prioritarias": (
        tabla_candidatas_prioritarias
    ),
    "resumen_reglas_padre": (
        resumen_reglas_padre
    )
}

ruta_estado_actual = (
    f"{carpeta_experimento}/"
    f"estado_experimento_semilla_{semilla_actual}.pkl"
)

joblib.dump(
    estado_experimento_actual,
    ruta_estado_actual
)

print(f"Estado guardado en:\n{ruta_estado_actual}")

In [ ]:
# ============================================================
# PREPARAR VALIDACIÓN CON NUEVAS SEMILLAS
# ============================================================

import inspect

SEMILLAS_VALIDACION = [101, 102, 103]
PARTICULAS_VALIDACION = particulas_actuales
ITERACIONES_VALIDACION = iteraciones_actuales

SOPORTES_VALIDACION = [0.0005, 0.001, 0.002]
CONFIANZAS_VALIDACION = [0.60, 0.70, 0.80]

print(
    f"Partículas: {PARTICULAS_VALIDACION}\n"
    f"Iteraciones: {ITERACIONES_VALIDACION}\n"
    f"Semillas: {SEMILLAS_VALIDACION}\n"
    f"Reglas dominantes: {ids_validacion}"
)

print("\nFirma de ejecutar_pso_anomalo:")
print(inspect.signature(ejecutar_pso_anomalo))

In [ ]:
CARGAR_ESTADO_PREVIO = False

if CARGAR_ESTADO_PREVIO:
    import joblib
    
    carpeta_experimento = (
        carpeta_base / "experimento_anomalias_35_particulas"
    )

    ruta_estado = (
        f"{carpeta_experimento}/"
        "estado_experimento_semilla_55.pkl"
    )

    estado_experimento_actual = joblib.load(ruta_estado)

    tabla_anomalias_no_redundantes = (
        estado_experimento_actual[
            "tabla_anomalias_no_redundantes"
        ]
    )

    resultado_redundancia_anomalias = (
        estado_experimento_actual[
            "resultado_redundancia_anomalias"
        ]
    )

    tabla_candidatas_prioritarias = (
        estado_experimento_actual[
            "tabla_candidatas_prioritarias"
        ]
    )

    resumen_reglas_padre = (
        estado_experimento_actual[
            "resumen_reglas_padre"
        ]
    )

    print("Estado del experimento cargado correctamente.")
else:
    print("No se carga un estado historico; se conservan los resultados de esta ejecucion.")


## 16. Validacion de estabilidad con nuevas semillas

La validacion repite la busqueda para las reglas dominantes seleccionadas usando las semillas $101$, $102$ y $103$. La semilla $55$ corresponde a la ejecucion base. Las familias se agrupan por regla dominante y mediante Jaccard difuso, y despues se clasifican segun el numero de semillas en las que aparecen.


In [ ]:
# ============================================================
# VALIDAR ANOMALÍAS CON NUEVAS SEMILLAS
# ============================================================

IDS_VALIDACION = ["RD_0810", "RD_0703", "RD_0467"]
SEMILLA_BASE = 55
SEMILLAS_VALIDACION = [101, 102, 103]

PARTICULAS_VALIDACION = 35
ITERACIONES_VALIDACION = 40
PROBABILIDAD_EXPLORACION_VALIDACION = 0.10

SOPORTES_VALIDACION = [0.0005, 0.001, 0.002]
CONFIANZAS_VALIDACION = [0.60, 0.70, 0.80]

requeridos = [
    "X_datos",
    "y_objetivo",
    "catalogo_dominantes_principal",
    "reglas_dominantes_principales",
    "preparar_contexto_dominante",
    "ejecutar_pso_anomalo"
]

faltantes = [
    nombre for nombre in requeridos
    if nombre not in globals()
]

if faltantes:
    raise NameError(
        "Ejecuta primero las celdas anteriores. "
        f"Faltan: {faltantes}"
    )

indice_por_regla_id = {
    str(regla_id): indice
    for indice, regla_id in enumerate(
        catalogo_dominantes_principal["regla_id"]
    )
}

ids_faltantes = [
    regla_id for regla_id in IDS_VALIDACION
    if regla_id not in indice_por_regla_id
]

if ids_faltantes:
    raise ValueError(
        f"No se encontraron estas reglas: {ids_faltantes}"
    )

carpeta_validacion = (
    f"{carpeta_experimento}/"
    "validacion_semillas_35p_40i"
)

os.makedirs(carpeta_validacion, exist_ok=True)

ruta_checkpoint_validacion = (
    f"{carpeta_validacion}/checkpoint_validacion.pkl"
)

ruta_resultados_validacion = (
    f"{carpeta_validacion}/resultados_nuevas_semillas.pkl"
)


def compactar_resultado_anomalo(resultado):
    """Elimina activaciones extensas antes de guardar."""
    resultado_compacto = copy.deepcopy(resultado)
    metricas = resultado_compacto.get("metricas", {})

    for clave in list(metricas):
        if "activacion" in clave.lower():
            metricas.pop(clave)

    return resultado_compacto


def guardar_checkpoint_validacion():
    estado = {
        "configuracion": {
            "ids_validacion": IDS_VALIDACION,
            "semilla_base": SEMILLA_BASE,
            "semillas_validacion": SEMILLAS_VALIDACION,
            "particulas": PARTICULAS_VALIDACION,
            "iteraciones": ITERACIONES_VALIDACION,
            "soportes": SOPORTES_VALIDACION,
            "confianzas": CONFIANZAS_VALIDACION,
            "probabilidad_exploracion": (
                PROBABILIDAD_EXPLORACION_VALIDACION
            )
        },
        "ejecuciones": registros_validacion,
        "candidatas": candidatas_validacion
    }

    joblib.dump(
        estado,
        ruta_checkpoint_validacion
    )


if os.path.exists(ruta_checkpoint_validacion):
    estado_checkpoint = joblib.load(
        ruta_checkpoint_validacion
    )

    registros_validacion = estado_checkpoint.get(
        "ejecuciones",
        []
    )

    candidatas_validacion = estado_checkpoint.get(
        "candidatas",
        []
    )

    print(
        f"Checkpoint recuperado: "
        f"{len(registros_validacion)} ejecuciones registradas."
    )
else:
    registros_validacion = []
    candidatas_validacion = []

claves_completadas = {
    (
        fila["regla_id_dominante"],
        int(fila["semilla"]),
        round(float(fila["soporte_minimo"]), 7)
    )
    for fila in registros_validacion
}

total_ejecuciones = (
    len(IDS_VALIDACION)
    * len(SEMILLAS_VALIDACION)
    * len(SOPORTES_VALIDACION)
)

for regla_id in IDS_VALIDACION:
    indice_dominante = indice_por_regla_id[regla_id]

    regla_dominante = (
        reglas_dominantes_principales[
            indice_dominante
        ]
    )

    contexto = preparar_contexto_dominante(
        regla_dominante
    )

    activacion_noY = (
        y_objetivo.to_numpy()
        == contexto["valor_anomalo"]
    ).astype(float)

    soporte_maximo_anomalo = (
        np.minimum(
            contexto["activacion_X"],
            activacion_noY
        ).sum()
        / len(y_objetivo)
    )

    for semilla in SEMILLAS_VALIDACION:
        for soporte_minimo in SOPORTES_VALIDACION:
            clave_ejecucion = (
                regla_id,
                semilla,
                round(soporte_minimo, 7)
            )

            if clave_ejecucion in claves_completadas:
                continue

            if soporte_maximo_anomalo < soporte_minimo:
                registros_validacion.append({
                    "regla_id_dominante": regla_id,
                    "semilla": semilla,
                    "soporte_minimo": soporte_minimo,
                    "soporte_maximo_anomalo": (
                        soporte_maximo_anomalo
                    ),
                    "estado": "no_factible",
                    "candidatas": 0,
                    "gbest_fitness": np.nan,
                    "gbest_soporte": np.nan,
                    "gbest_confianza": np.nan
                })

                claves_completadas.add(
                    clave_ejecucion
                )

                guardar_checkpoint_validacion()
                continue

            resultado_pso = ejecutar_pso_anomalo(
                dataframe=X_datos,
                objetivo=y_objetivo,
                activacion_X=contexto["activacion_X"],
                variables_X=contexto["variables_X"],
                valor_consecuente_dominante=(
                    contexto["valor_consecuente"]
                ),
                metodo_similitud=(
                    contexto["metodo_similitud"]
                ),
                soporte_minimo=soporte_minimo,
                numero_particulas=(
                    PARTICULAS_VALIDACION
                ),
                numero_iteraciones=(
                    ITERACIONES_VALIDACION
                ),
                probabilidad_exploracion=(
                    PROBABILIDAD_EXPLORACION_VALIDACION
                ),
                semilla=semilla
            )

            candidatas_ejecucion = 0

            for indice_particula, resultado_regla in enumerate(
                resultado_pso.get(
                    "pbest_resultados",
                    []
                ),
                start=1
            ):
                if not resultado_regla:
                    continue

                metricas = resultado_regla.get(
                    "metricas"

                )or {}

                soporte_anomalo = float(
                    metricas.get(
                        "soporte_anomalo",
                        0.0
                    )
                )

                confianza_anomala = float(
                    metricas.get(
                        "confianza_anomala",
                        0.0
                    )
                )

                cf_directo = float(
                    metricas.get(
                        "cf_Z_hacia_noY",
                        metricas.get("cf_directo", 0.0)
                    )
                )

                cf_inverso = float(
                    metricas.get(
                        "cf_noY_hacia_Z",
                        metricas.get("cf_inverso", 0.0)
                    )
                )

                condiciones_Z = resultado_regla.get(
                    "condiciones_Z",
                    []
                )

                confianzas_superadas = [
                    confianza
                    for confianza in CONFIANZAS_VALIDACION
                    if confianza_anomala >= confianza
                ]

                cumple_criterios = (
                    resultado_regla.get("fitness", 0.0) > 0
                    and soporte_anomalo >= soporte_minimo
                    and cf_directo > 0
                    and cf_inverso > 0
                    and 1 <= len(condiciones_Z) <= 3
                    and bool(confianzas_superadas)
                )

                if not cumple_criterios:
                    continue

                candidatas_ejecucion += 1

                candidatas_validacion.append({
                    "regla_id_dominante": regla_id,
                    "indice_dominante": indice_dominante,
                    "semilla": semilla,
                    "particula": indice_particula,
                    "soporte_busqueda": soporte_minimo,
                    "confianzas_superadas": (
                        confianzas_superadas
                    ),
                    "metodo_similitud": (
                        contexto["metodo_similitud"]
                    ),
                    "resultado": (
                        compactar_resultado_anomalo(
                            resultado_regla
                        )
                    )
                })

            gbest = resultado_pso.get(
                "gbest_resultado",
                {}
            )

            metricas_gbest = gbest.get(
                "metricas"
            )or {}

            registros_validacion.append({
                "regla_id_dominante": regla_id,
                "semilla": semilla,
                "soporte_minimo": soporte_minimo,
                "soporte_maximo_anomalo": (
                    soporte_maximo_anomalo
                ),
                "estado": "ejecutada",
                "candidatas": candidatas_ejecucion,
                "gbest_fitness": resultado_pso.get(
                    "gbest_fitness",
                    gbest.get("fitness", np.nan)
                ),
                "gbest_soporte": metricas_gbest.get(
                    "soporte_anomalo",
                    np.nan
                ),
                "gbest_confianza": metricas_gbest.get(
                    "confianza_anomala",
                    np.nan
                )
            })

            claves_completadas.add(
                clave_ejecucion
            )

            guardar_checkpoint_validacion()

            procesadas = len(claves_completadas)

            if procesadas % 3 == 0 or procesadas == total_ejecuciones:
                print(
                    f"Procesadas {procesadas}/{total_ejecuciones}. "
                    f"Candidatas acumuladas: "
                    f"{len(candidatas_validacion)}"
                )

            del resultado_pso
            gc.collect()

tabla_ejecuciones_validacion = pd.DataFrame(
    registros_validacion
)

resultado_validacion = {
    "configuracion": {
        "ids_validacion": IDS_VALIDACION,
        "semilla_base": SEMILLA_BASE,
        "semillas_nuevas": SEMILLAS_VALIDACION,
        "particulas": PARTICULAS_VALIDACION,
        "iteraciones": ITERACIONES_VALIDACION,
        "soportes": SOPORTES_VALIDACION,
        "confianzas": CONFIANZAS_VALIDACION
    },
    "ejecuciones": registros_validacion,
    "candidatas": candidatas_validacion
}

joblib.dump(
    resultado_validacion,
    ruta_resultados_validacion
)

tabla_ejecuciones_validacion.to_csv(
    f"{carpeta_validacion}/"
    "resumen_ejecuciones_validacion.csv",
    index=False
)

print(
    f"\nValidación terminada.\n"
    f"Ejecuciones registradas: "
    f"{len(tabla_ejecuciones_validacion)}\n"
    f"Candidatas nuevas: "
    f"{len(candidatas_validacion)}\n"
    f"Resultados: {ruta_resultados_validacion}"
)

display(
    tabla_ejecuciones_validacion
    .groupby(
        [
            "regla_id_dominante",
            "estado"
        ],
        dropna=False
    )
    .agg(
        ejecuciones=("semilla", "count"),
        candidatas=("candidatas", "sum"),
        mejor_confianza=("gbest_confianza", "max"),
        mejor_fitness=("gbest_fitness", "max")
    )
    .reset_index()
    .round(6)
)

In [ ]:
# ============================================================
# AGRUPAR FAMILIAS Y CALCULAR ESTABILIDAD
# ============================================================

UMBRAL_JACCARD_VALIDACION = 0.85

if "resultado_redundancia_anomalias" not in globals():
    raise NameError(
        "Carga primero el estado del experimento inicial."
    )

candidatas_semilla_base = []

for candidata in resultado_redundancia_anomalias[
    "reglas_no_redundantes"
]:
    regla_id = candidata["regla_id_dominante"]

    if regla_id not in IDS_VALIDACION:
        continue

    candidata_base = copy.deepcopy(candidata)
    metricas = candidata_base["resultado"]["metricas"]
    confianza = float(metricas["confianza_anomala"])

    candidata_base["semilla"] = SEMILLA_BASE
    candidata_base["origen"] = "experimento_inicial"
    candidata_base["confianzas_superadas"] = [
        umbral
        for umbral in CONFIANZAS_VALIDACION
        if confianza >= umbral
    ]

    candidatas_semilla_base.append(
        candidata_base
    )

candidatas_todas_semillas = (
    candidatas_semilla_base
    + copy.deepcopy(candidatas_validacion)
)

candidatas_todas_semillas.sort(
    key=lambda candidata: (
        candidata["resultado"]["metricas"].get(
            "confianza_anomala",
            0.0
        ),
        candidata["resultado"].get(
            "fitness",
            0.0
        )
    ),
    reverse=True
)


def jaccard_difuso_validacion(
    activacion_a,
    activacion_b
):
    interseccion = np.minimum(
        activacion_a,
        activacion_b
    ).sum()

    union = np.maximum(
        activacion_a,
        activacion_b
    ).sum()

    return float(
        interseccion / union
    ) if union > 0 else 0.0


def recuperar_soportes_candidata(candidata):
    valores = candidata.get(
        "soportes_familia",
        {candidata.get("soporte_busqueda")}
    )

    if not isinstance(
        valores,
        (set, list, tuple)
    ):
        valores = {valores}

    return {
        float(valor)
        for valor in valores
        if valor is not None
    }


contextos_validacion = {}
familias_por_grupo = {}
activaciones_por_grupo = {}
comparaciones_jaccard = 0

for candidata in candidatas_todas_semillas:
    regla_id = candidata["regla_id_dominante"]
    indice_dominante = indice_por_regla_id[regla_id]

    if regla_id not in contextos_validacion:
        contextos_validacion[regla_id] = (
            preparar_contexto_dominante(
                reglas_dominantes_principales[
                    indice_dominante
                ]
            )
        )

    contexto = contextos_validacion[regla_id]

    metodo = candidata.get(
        "metodo_similitud",
        contexto["metodo_similitud"]
    )

    clave_grupo = (
        regla_id,
        metodo
    )

    if clave_grupo not in familias_por_grupo:
        familias_por_grupo[clave_grupo] = []
        activaciones_por_grupo[clave_grupo] = []

    condiciones_Z = candidata[
        "resultado"
    ]["condiciones_Z"]

    activacion_Z = calcular_activacion_antecedente(
        X_datos,
        condiciones_Z,
        metodo
    )

    activacion_XZ = np.minimum(
        contexto["activacion_X"],
        activacion_Z
    ).astype(np.float32)

    familias = familias_por_grupo[
        clave_grupo
    ]

    activaciones = activaciones_por_grupo[
        clave_grupo
    ]

    indice_familia = None
    mayor_jaccard = 0.0

    for indice, activacion_representante in enumerate(
        activaciones
    ):
        comparaciones_jaccard += 1

        similitud = jaccard_difuso_validacion(
            activacion_XZ,
            activacion_representante
        )

        if similitud > mayor_jaccard:
            mayor_jaccard = similitud

        if similitud >= UMBRAL_JACCARD_VALIDACION:
            indice_familia = indice
            break

    semilla = int(candidata["semilla"])
    confianzas = set(
        candidata["confianzas_superadas"]
    )

    soportes = recuperar_soportes_candidata(
        candidata
    )

    peso_familia = int(
        candidata.get(
            "miembros_familia_jaccard",
            1
        )
    )

    if indice_familia is None:
        semillas_por_confianza = {
            umbral: set()
            for umbral in CONFIANZAS_VALIDACION
        }

        for umbral in confianzas:
            semillas_por_confianza[
                umbral
            ].add(semilla)

        familias.append({
            "representante": copy.deepcopy(candidata),
            "semillas": {semilla},
            "semillas_por_confianza": (
                semillas_por_confianza
            ),
            "soportes_busqueda": soportes,
            "miembros_familia": peso_familia,
            "jaccard_maximo": 0.0
        })

        activaciones.append(
            activacion_XZ
        )
    else:
        familia = familias[indice_familia]

        familia["semillas"].add(
            semilla
        )

        familia["soportes_busqueda"].update(
            soportes
        )

        familia["miembros_familia"] += (
            peso_familia
        )

        familia["jaccard_maximo"] = max(
            familia["jaccard_maximo"],
            mayor_jaccard
        )

        for umbral in confianzas:
            familia[
                "semillas_por_confianza"
            ][umbral].add(semilla)

familias_anomalas = [
    familia
    for familias in familias_por_grupo.values()
    for familia in familias
]

familias_anomalas.sort(
    key=lambda familia: (
        len(
            familia[
                "semillas_por_confianza"
            ][0.70]
        ),
        len(familia["semillas"]),
        familia[
            "representante"
        ]["resultado"]["metricas"].get(
            "confianza_anomala",
            0.0
        ),
        familia[
            "representante"
        ]["resultado"].get(
            "fitness",
            0.0
        )
    ),
    reverse=True
)

total_semillas = 1 + len(
    SEMILLAS_VALIDACION
)

filas_familias = []

for posicion, familia in enumerate(
    familias_anomalas,
    start=1
):
    representante = familia[
        "representante"
    ]

    regla_id = representante[
        "regla_id_dominante"
    ]

    resultado = representante[
        "resultado"
    ]

    metricas = resultado[
        "metricas"
    ]

    numero_semillas = len(
        familia["semillas"]
    )

    semillas_conf_60 = len(
        familia[
            "semillas_por_confianza"
        ][0.60]
    )

    semillas_conf_70 = len(
        familia[
            "semillas_por_confianza"
        ][0.70]
    )

    semillas_conf_80 = len(
        familia[
            "semillas_por_confianza"
        ][0.80]
    )

    if semillas_conf_60 == total_semillas:
        nivel_estabilidad = "alta"
    elif semillas_conf_60 >= 2:
        nivel_estabilidad = "media"
    else:
        nivel_estabilidad = "exploratoria"

    fila_catalogo = (
        catalogo_dominantes_principal.iloc[
            indice_por_regla_id[regla_id]
        ]
    )

    filas_familias.append({
        "posicion": posicion,
        "regla_id_dominante": regla_id,
        "regla_dominante": fila_catalogo["regla"],
        "metodo_similitud": representante[
            "metodo_similitud"
        ],
        "Z": describir_condiciones_Z(
            resultado["condiciones_Z"]
        ),
        "numero_condiciones_Z": len(
            resultado["condiciones_Z"]
        ),
        "fitness": resultado["fitness"],
        "soporte_anomalo": metricas[
            "soporte_anomalo"
        ],
        "confianza_anomala": metricas[
            "confianza_anomala"
        ],
        "confianza_inversa": metricas[
            "confianza_inversa"
        ],
        "cf_directo": metricas[
            "cf_Z_hacia_noY"
        ],
        "cf_inverso": metricas[
            "cf_noY_hacia_Z"
        ],
        "numero_semillas": numero_semillas,
        "semillas": ", ".join(
            str(semilla)
            for semilla in sorted(
                familia["semillas"]
            )
        ),
        "semillas_conf_60": semillas_conf_60,
        "semillas_conf_70": semillas_conf_70,
        "semillas_conf_80": semillas_conf_80,
        "estable_conf_60": semillas_conf_60 >= 2,
        "estable_conf_70": semillas_conf_70 >= 2,
        "nivel_estabilidad": nivel_estabilidad,
        "soportes_busqueda": ", ".join(
            f"{soporte:.4f}"
            for soporte in sorted(
                familia["soportes_busqueda"]
            )
        ),
        "miembros_familia": familia[
            "miembros_familia"
        ],
        "jaccard_maximo": familia[
            "jaccard_maximo"
        ]
    })

tabla_familias_anomalas = pd.DataFrame(
    filas_familias
)

tabla_familias_estables = (
    tabla_familias_anomalas[
        tabla_familias_anomalas[
            "estable_conf_60"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

resultado_final_validacion = {
    "configuracion": resultado_validacion[
        "configuracion"
    ],
    "familias_anomalas": familias_anomalas,
    "tabla_familias": tabla_familias_anomalas,
    "tabla_familias_estables": (
        tabla_familias_estables
    ),
    "umbral_jaccard": (
        UMBRAL_JACCARD_VALIDACION
    ),
    "comparaciones_jaccard": (
        comparaciones_jaccard
    )
}

ruta_final_validacion = (
    f"{carpeta_validacion}/"
    "resultado_final_validacion.pkl"
)

joblib.dump(
    resultado_final_validacion,
    ruta_final_validacion
)

tabla_familias_anomalas.to_csv(
    f"{carpeta_validacion}/"
    "familias_anomalas_todas.csv",
    index=False
)

tabla_familias_estables.to_csv(
    f"{carpeta_validacion}/"
    "familias_anomalas_estables.csv",
    index=False
)

print(
    f"Familias encontradas: "
    f"{len(tabla_familias_anomalas)}\n"
    f"Estables con confianza >= 0.60: "
    f"{tabla_familias_anomalas['estable_conf_60'].sum()}\n"
    f"Estables con confianza >= 0.70: "
    f"{tabla_familias_anomalas['estable_conf_70'].sum()}\n"
    f"Comparaciones Jaccard: "
    f"{comparaciones_jaccard}\n"
    f"Resultados: {ruta_final_validacion}"
)

print("\nResumen de estabilidad:")

display(
    tabla_familias_anomalas
    .groupby("nivel_estabilidad")
    .agg(
        familias=("posicion", "count"),
        confianza_promedio=(
            "confianza_anomala",
            "mean"
        ),
        soporte_promedio=(
            "soporte_anomalo",
            "mean"
        ),
        fitness_promedio=("fitness", "mean")
    )
    .reset_index()
    .round(6)
)

print("\nFamilias estables:")

display(
    tabla_familias_estables[
        [
            "regla_id_dominante",
            "regla_dominante",
            "Z",
            "soporte_anomalo",
            "confianza_anomala",
            "confianza_inversa",
            "fitness",
            "semillas_conf_60",
            "semillas_conf_70",
            "nivel_estabilidad"
        ]
    ].round(6)
)

## 18. Catalogos y resultados finales

Se generan el catalogo completo de familias anomalas, el catalogo ampliado y el catalogo principal de reglas estables. Las graficas y tablas describen $R_d$, $Z$, $Supp_a$, las confianzas condicionadas, $CF_X$ y la estabilidad; no sustituyen la validacion estadistica o clinica.


In [ ]:
# ============================================================
# GENERAR RESULTADOS Y GRÁFICAS FINALES
# ============================================================

if "tabla_familias_anomalas" not in globals():
    raise NameError(
        "Ejecuta primero la agrupación de familias anómalas."
    )

tabla_resultados = tabla_familias_anomalas.copy()

if tabla_resultados.empty:
    raise ValueError("No existen familias anómalas para analizar.")

# 1. Asegurar columnas booleanas
for columna in ["estable_conf_60", "estable_conf_70"]:
    if not pd.api.types.is_bool_dtype(
        tabla_resultados[columna]
    ):
        tabla_resultados[columna] = (
            tabla_resultados[columna]
            .astype(str)
            .str.lower()
            .eq("true")
        )

# 2. Incorporar métricas de las reglas dominantes
metricas_dominantes = (
    catalogo_dominantes_principal[
        [
            "regla_id",
            "confianza",
            "soporte_XY",
            "lift",
            "factor_certeza"
        ]
    ]
    .rename(columns={
        "regla_id": "regla_id_dominante",
        "confianza": "confianza_dominante",
        "soporte_XY": "soporte_dominante",
        "lift": "lift_dominante",
        "factor_certeza": "cf_dominante"
    })
)

tabla_resultados = tabla_resultados.merge(
    metricas_dominantes,
    on="regla_id_dominante",
    how="left"
)

# 3. Calcular masa equivalente
tabla_resultados["masa_equivalente"] = (
    tabla_resultados["soporte_anomalo"]
    * len(y_objetivo)
)

# 4. Ordenar y asignar identificadores
tabla_resultados = (
    tabla_resultados
    .sort_values(
        [
            "estable_conf_70",
            "estable_conf_60",
            "semillas_conf_70",
            "semillas_conf_60",
            "confianza_anomala",
            "fitness"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

tabla_resultados.insert(
    0,
    "anomalia_id",
    [
        f"RA_{indice:03d}"
        for indice in range(
            1,
            len(tabla_resultados) + 1
        )
    ]
)

# 5. Crear los tres catálogos
catalogo_anomalias_completo = (
    tabla_resultados.copy()
)

catalogo_anomalias_ampliado = (
    tabla_resultados[
        tabla_resultados["estable_conf_60"]
    ]
    .copy()
    .reset_index(drop=True)
)

catalogo_anomalias_principal = (
    tabla_resultados[
        tabla_resultados["estable_conf_70"]
    ]
    .copy()
    .reset_index(drop=True)
)

# 6. Resumen de los catálogos
def resumir_catalogo(nombre, catalogo):
    return {
        "catalogo": nombre,
        "familias": len(catalogo),
        "reglas_dominantes": (
            catalogo["regla_id_dominante"].nunique()
        ),
        "confianza_promedio": (
            catalogo["confianza_anomala"].mean()
            if len(catalogo) else np.nan
        ),
        "confianza_maxima": (
            catalogo["confianza_anomala"].max()
            if len(catalogo) else np.nan
        ),
        "soporte_promedio": (
            catalogo["soporte_anomalo"].mean()
            if len(catalogo) else np.nan
        ),
        "masa_equivalente_promedio": (
            catalogo["masa_equivalente"].mean()
            if len(catalogo) else np.nan
        ),
        "fitness_promedio": (
            catalogo["fitness"].mean()
            if len(catalogo) else np.nan
        )
    }


resumen_catalogos = pd.DataFrame([
    resumir_catalogo(
        "completo",
        catalogo_anomalias_completo
    ),
    resumir_catalogo(
        "estable_conf_60",
        catalogo_anomalias_ampliado
    ),
    resumir_catalogo(
        "estable_conf_70",
        catalogo_anomalias_principal
    )
])

# 7. Resumen por regla dominante
resumen_por_regla = (
    tabla_resultados
    .groupby("regla_id_dominante")
    .agg(
        familias_totales=(
            "anomalia_id",
            "count"
        ),
        familias_estables_60=(
            "estable_conf_60",
            "sum"
        ),
        familias_estables_70=(
            "estable_conf_70",
            "sum"
        ),
        mejor_confianza=(
            "confianza_anomala",
            "max"
        ),
        mejor_soporte=(
            "soporte_anomalo",
            "max"
        ),
        mejor_fitness=(
            "fitness",
            "max"
        ),
        maximo_semillas=(
            "numero_semillas",
            "max"
        )
    )
    .sort_values(
        [
            "familias_estables_70",
            "familias_estables_60",
            "mejor_confianza"
        ],
        ascending=False
    )
    .reset_index()
)

# 8. Resumen de condiciones Z repetidas
resumen_condiciones_Z = (
    tabla_resultados
    .groupby("Z")
    .agg(
        familias=("anomalia_id", "count"),
        reglas_dominantes=(
            "regla_id_dominante",
            "nunique"
        ),
        familias_estables_60=(
            "estable_conf_60",
            "sum"
        ),
        familias_estables_70=(
            "estable_conf_70",
            "sum"
        ),
        confianza_maxima=(
            "confianza_anomala",
            "max"
        ),
        soporte_maximo=(
            "soporte_anomalo",
            "max"
        )
    )
    .sort_values(
        [
            "familias_estables_70",
            "reglas_dominantes",
            "confianza_maxima"
        ],
        ascending=False
    )
    .reset_index()
)

# 9. Definir carpeta de resultados
carpeta_resultados_finales = (
    f"{carpeta_validacion}/resultados_finales"
)

carpeta_graficas = (
    f"{carpeta_resultados_finales}/graficas"
)

os.makedirs(
    carpeta_graficas,
    exist_ok=True
)

# 10. Guardar catálogos y resúmenes
catalogo_anomalias_completo.to_csv(
    f"{carpeta_resultados_finales}/"
    "catalogo_anomalias_completo.csv",
    index=False
)

catalogo_anomalias_ampliado.to_csv(
    f"{carpeta_resultados_finales}/"
    "catalogo_anomalias_estables_conf_60.csv",
    index=False
)

catalogo_anomalias_principal.to_csv(
    f"{carpeta_resultados_finales}/"
    "catalogo_anomalias_estables_conf_70.csv",
    index=False
)

resumen_catalogos.to_csv(
    f"{carpeta_resultados_finales}/"
    "resumen_catalogos.csv",
    index=False
)

resumen_por_regla.to_csv(
    f"{carpeta_resultados_finales}/"
    "resumen_por_regla_dominante.csv",
    index=False
)

resumen_condiciones_Z.to_csv(
    f"{carpeta_resultados_finales}/"
    "resumen_condiciones_Z.csv",
    index=False
)

# ============================================================
# GRÁFICA 1. REDUCCIÓN DE FAMILIAS
# ============================================================

cantidad_total = len(
    catalogo_anomalias_completo
)

cantidad_estables_60 = len(
    catalogo_anomalias_ampliado
)

cantidad_estables_70 = len(
    catalogo_anomalias_principal
)

cantidad_una_semilla = int(
    (~tabla_resultados["estable_conf_60"]).sum()
)

etiquetas = [
    "Total",
    "Estables\nconf. ≥ 0.60",
    "Estables\nconf. ≥ 0.70",
    "Una sola\nsemilla"
]

cantidades = [
    cantidad_total,
    cantidad_estables_60,
    cantidad_estables_70,
    cantidad_una_semilla
]

fig, ax = plt.subplots(figsize=(9, 5))

barras = ax.bar(
    etiquetas,
    cantidades,
    color=[
        "steelblue",
        "seagreen",
        "darkorange",
        "gray"
    ]
)

ax.bar_label(
    barras,
    padding=3
)

ax.set_ylabel("Número de familias")
ax.set_title("Selección de familias anómalas")
ax.set_ylim(
    0,
    max(cantidades) * 1.18
)
ax.grid(
    axis="y",
    alpha=0.25
)

fig.tight_layout()

fig.savefig(
    f"{carpeta_graficas}/"
    "01_seleccion_familias.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ============================================================
# GRÁFICA 2. CONFIANZA DOMINANTE Y ANÓMALA
# ============================================================

if not catalogo_anomalias_principal.empty:
    datos_principales = (
        catalogo_anomalias_principal.copy()
    )

    etiquetas_principales = (
        datos_principales[
            "regla_id_dominante"
        ]
        + "\n"
        + datos_principales["anomalia_id"]
    )

    posiciones = np.arange(
        len(datos_principales)
    )

    ancho = 0.36

    fig, ax = plt.subplots(figsize=(9, 5))

    barras_dominantes = ax.bar(
        posiciones - ancho / 2,
        datos_principales[
            "confianza_dominante"
        ],
        ancho,
        label="Regla dominante"
    )

    barras_anomalas = ax.bar(
        posiciones + ancho / 2,
        datos_principales[
            "confianza_anomala"
        ],
        ancho,
        label="Regla anómala"
    )

    ax.bar_label(
        barras_dominantes,
        fmt="%.3f",
        padding=3
    )

    ax.bar_label(
        barras_anomalas,
        fmt="%.3f",
        padding=3
    )

    ax.set_xticks(
        posiciones,
        etiquetas_principales
    )

    ax.set_ylabel("Confianza")
    ax.set_title(
        "Confianza de las reglas dominantes "
        "y sus anomalías estables"
    )
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(
        axis="y",
        alpha=0.25
    )

    fig.tight_layout()

    fig.savefig(
        f"{carpeta_graficas}/"
        "02_confianza_dominante_anomala.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

# ============================================================
# GRÁFICA 3. SOPORTE, CONFIANZA Y ESTABILIDAD
# ============================================================

colores_estabilidad = {
    "alta": "seagreen",
    "media": "darkorange",
    "exploratoria": "gray"
}

fig, ax = plt.subplots(figsize=(10, 6))

for nivel in [
    "alta",
    "media",
    "exploratoria"
]:
    subconjunto = tabla_resultados[
        tabla_resultados[
            "nivel_estabilidad"
        ] == nivel
    ]

    if subconjunto.empty:
        continue

    ax.scatter(
        subconjunto["masa_equivalente"],
        subconjunto["confianza_anomala"],
        s=(
            45
            + 35
            * subconjunto["numero_semillas"]
        ),
        alpha=0.75,
        color=colores_estabilidad[nivel],
        label=nivel.capitalize()
    )

for _, fila in catalogo_anomalias_principal.iterrows():
    ax.annotate(
        (
            f"{fila['regla_id_dominante']}\n"
            f"{fila['anomalia_id']}"
        ),
        (
            fila["masa_equivalente"],
            fila["confianza_anomala"]
        ),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=9
    )

ax.axhline(
    0.60,
    linestyle="--",
    color="gray",
    alpha=0.6,
    label="Confianza 0.60"
)

ax.axhline(
    0.70,
    linestyle=":",
    color="darkred",
    alpha=0.7,
    label="Confianza 0.70"
)

ax.set_xlabel(
    "Masa equivalente de casos anómalos"
)

ax.set_ylabel(
    "Confianza anómala"
)

ax.set_title(
    "Soporte, confianza y estabilidad "
    "de las familias anómalas"
)

ax.grid(alpha=0.25)
ax.legend()

fig.tight_layout()

fig.savefig(
    f"{carpeta_graficas}/"
    "03_soporte_confianza_estabilidad.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# 11. Guardar estructura final
resultados_finales_anomalias = {
    "catalogo_completo": (
        catalogo_anomalias_completo
    ),
    "catalogo_estable_conf_60": (
        catalogo_anomalias_ampliado
    ),
    "catalogo_estable_conf_70": (
        catalogo_anomalias_principal
    ),
    "resumen_catalogos": (
        resumen_catalogos
    ),
    "resumen_por_regla": (
        resumen_por_regla
    ),
    "resumen_condiciones_Z": (
        resumen_condiciones_Z
    ),
    "configuracion": {
        "semillas": [
            SEMILLA_BASE,
            *SEMILLAS_VALIDACION
        ],
        "particulas": (
            PARTICULAS_VALIDACION
        ),
        "iteraciones": (
            ITERACIONES_VALIDACION
        ),
        "soportes": (
            SOPORTES_VALIDACION
        ),
        "confianzas": (
            CONFIANZAS_VALIDACION
        ),
        "umbral_jaccard": (
            UMBRAL_JACCARD_VALIDACION
        )
    }
}

ruta_resultados_finales = (
    f"{carpeta_resultados_finales}/"
    "resultados_finales_anomalias.pkl"
)

joblib.dump(
    resultados_finales_anomalias,
    ruta_resultados_finales
)

# 12. Mostrar resultados finales
print("=" * 80)
print("RESULTADOS FINALES DE REGLAS ANÓMALAS")
print("=" * 80)

print(
    f"\nFamilias encontradas: "
    f"{cantidad_total}"
)

print(
    f"Familias estables con confianza >= 0.60: "
    f"{cantidad_estables_60}"
)

print(
    f"Familias estables con confianza >= 0.70: "
    f"{cantidad_estables_70}"
)

print(
    f"Familias encontradas en una sola semilla: "
    f"{cantidad_una_semilla}"
)

print(
    f"Reglas dominantes con anomalía estable >= 0.70: "
    f"{catalogo_anomalias_principal['regla_id_dominante'].nunique()}"
)

print("\nResumen de catálogos:")
display(
    resumen_catalogos.round(6)
)

print("\nResultados por regla dominante:")
display(
    resumen_por_regla.round(6)
)

print("\nCatálogo principal:")
display(
    catalogo_anomalias_principal[
        [
            "anomalia_id",
            "regla_id_dominante",
            "regla_dominante",
            "Z",
            "confianza_dominante",
            "confianza_anomala",
            "confianza_inversa",
            "soporte_anomalo",
            "masa_equivalente",
            "fitness",
            "semillas_conf_70",
            "nivel_estabilidad"
        ]
    ].round(6)
)

print(
    f"\nResultados guardados en:\n"
    f"{carpeta_resultados_finales}"
)

In [ ]:
GENERAR_GRAFICAS_DOMINANTES = False

if GENERAR_GRAFICAS_DOMINANTES:
    # ============================================================
    # GENERAR GRÁFICAS DE REGLAS DOMINANTES
    # ============================================================

    carpeta_base = raiz_repositorio / "artifacts"

    rutas_experimentos_dominantes = {
        "50 partículas": (
            f"{carpeta_base}/"
            "resultados_reglas_dominantes.pkl"
        ),
        "100 partículas": (
            f"{carpeta_base}/"
            "experimento_100_particulas_soporte_001/"
            "resultados_reglas_dominantes.pkl"
        ),
        "500 partículas": (
            f"{carpeta_base}/"
            "experimento_500_particulas_soporte_001/"
            "resultados_reglas_dominantes.pkl"
        )
    }

    particulas_por_experimento = {
        "50 partículas": 50,
        "100 partículas": 100,
        "500 partículas": 500
    }

    archivos_faltantes = [
        ruta
        for ruta in rutas_experimentos_dominantes.values()
        if not os.path.exists(ruta)
    ]

    if archivos_faltantes:
        raise FileNotFoundError(
            f"No se encontraron estos archivos: {archivos_faltantes}"
        )

    resultados_dominantes = {
        nombre: joblib.load(ruta)
        for nombre, ruta
        in rutas_experimentos_dominantes.items()
    }


    def recuperar_catalogos_dominantes(resultado):
        """Recupera los catálogos completo, reproducible y estable."""
        catalogo_completo = resultado[
            "catalogo_completo"
        ].copy()

        catalogo_reproducible = resultado[
            "catalogo_reproducible"
        ].copy()

        if "nivel_estabilidad" in catalogo_reproducible:
            catalogo_alta = (
                catalogo_reproducible[
                    catalogo_reproducible[
                        "nivel_estabilidad"
                    ] == "alta"
                ]
                .copy()
                .reset_index(drop=True)
            )
        elif "numero_semillas" in catalogo_reproducible:
            total_semillas = max(
                catalogo_reproducible[
                    "numero_semillas"
                ].max(),
                1
            )

            catalogo_alta = (
                catalogo_reproducible[
                    catalogo_reproducible[
                        "numero_semillas"
                    ] == total_semillas
                ]
                .copy()
                .reset_index(drop=True)
            )
        else:
            raise ValueError(
                "No se encontró información de estabilidad."
            )

        return {
            "completo": catalogo_completo,
            "reproducible": catalogo_reproducible,
            "alta": catalogo_alta
        }


    catalogos_dominantes = {
        nombre: recuperar_catalogos_dominantes(
            resultado
        )
        for nombre, resultado
        in resultados_dominantes.items()
    }

    # 1. Crear resumen general
    filas_resumen = []

    for nombre, catalogos in catalogos_dominantes.items():
        completo = catalogos["completo"]
        reproducible = catalogos["reproducible"]
        alta = catalogos["alta"]

        numero_completo = len(completo)
        numero_reproducible = len(reproducible)
        numero_alta = len(alta)

        filas_resumen.append({
            "experimento": nombre,
            "particulas": (
                particulas_por_experimento[nombre]
            ),
            "reglas_finales": numero_completo,
            "reglas_reproducibles": (
                numero_reproducible
            ),
            "reglas_estabilidad_alta": numero_alta,
            "proporcion_reproducibles": (
                numero_reproducible / numero_completo
                if numero_completo else 0.0
            ),
            "proporcion_estabilidad_alta": (
                numero_alta / numero_completo
                if numero_completo else 0.0
            ),
            "reglas_por_particula": (
                numero_completo
                / particulas_por_experimento[nombre]
            ),
            "fitness_mediano_reproducible": (
                reproducible["fitness"].median()
            ),
            "confianza_mediana_reproducible": (
                reproducible["confianza"].median()
            ),
            "lift_mediano_reproducible": (
                reproducible["lift"].median()
            ),
            "fitness_maximo": completo["fitness"].max(),
            "confianza_maxima": (
                completo["confianza"].max()
            )
        })

    tabla_resumen_dominantes = (
        pd.DataFrame(filas_resumen)
        .sort_values("particulas")
        .reset_index(drop=True)
    )

    # 2. Resumen por longitud de X
    filas_longitud = []

    for nombre, catalogos in catalogos_dominantes.items():
        catalogo_alta = catalogos["alta"]
        total_alta = len(catalogo_alta)

        cantidades = (
            catalogo_alta[
                "numero_condiciones"
            ]
            .value_counts()
            .sort_index()
        )

        for longitud in range(1, 5):
            cantidad = int(
                cantidades.get(longitud, 0)
            )

            filas_longitud.append({
                "experimento": nombre,
                "particulas": (
                    particulas_por_experimento[nombre]
                ),
                "numero_condiciones": longitud,
                "cantidad": cantidad,
                "proporcion": (
                    cantidad / total_alta
                    if total_alta else 0.0
                )
            })

    tabla_longitud_dominantes = pd.DataFrame(
        filas_longitud
    )

    # 3. Reunir reglas reproducibles para comparaciones
    catalogos_reproducibles_unidos = []

    for nombre, catalogos in catalogos_dominantes.items():
        catalogo = (
            catalogos["reproducible"].copy()
        )

        catalogo["experimento"] = nombre
        catalogo["particulas"] = (
            particulas_por_experimento[nombre]
        )

        catalogos_reproducibles_unidos.append(
            catalogo
        )

    tabla_reproducibles_unida = pd.concat(
        catalogos_reproducibles_unidos,
        ignore_index=True
    )

    # 4. Resumen por consecuente y método
    resumen_metodo_consecuente = (
        tabla_reproducibles_unida
        .groupby(
            [
                "experimento",
                "consecuente",
                "metodo_similitud"
            ]
        )
        .agg(
            reglas=("regla", "count"),
            fitness_mediano=("fitness", "median"),
            confianza_mediana=(
                "confianza",
                "median"
            ),
            lift_mediano=("lift", "median")
        )
        .reset_index()
    )

    # 5. Preparar carpeta de salida
    carpeta_graficas_dominantes = (
        f"{carpeta_base}/"
        "comparacion_experimentos_50_100_500/"
        "graficas_reglas_dominantes"
    )

    os.makedirs(
        carpeta_graficas_dominantes,
        exist_ok=True
    )

    tabla_resumen_dominantes.to_csv(
        f"{carpeta_graficas_dominantes}/"
        "resumen_reglas_dominantes.csv",
        index=False
    )

    tabla_longitud_dominantes.to_csv(
        f"{carpeta_graficas_dominantes}/"
        "resumen_longitud_reglas_dominantes.csv",
        index=False
    )

    resumen_metodo_consecuente.to_csv(
        f"{carpeta_graficas_dominantes}/"
        "resumen_metodo_consecuente.csv",
        index=False
    )

    # ============================================================
    # GRÁFICA 1. CANTIDAD Y PROPORCIÓN DE REGLAS
    # ============================================================

    orden_experimentos = (
        tabla_resumen_dominantes["experimento"]
        .tolist()
    )

    posiciones = np.arange(
        len(tabla_resumen_dominantes)
    )

    ancho = 0.25

    fig, ejes = plt.subplots(
        1,
        2,
        figsize=(14, 5)
    )

    columnas_cantidad = [
        (
            "reglas_finales",
            "Finales"
        ),
        (
            "reglas_reproducibles",
            "Reproducibles"
        ),
        (
            "reglas_estabilidad_alta",
            "Estabilidad alta"
        )
    ]

    for desplazamiento, (
        columna,
        etiqueta
    ) in enumerate(columnas_cantidad):
        barras = ejes[0].bar(
            posiciones
            + (desplazamiento - 1) * ancho,
            tabla_resumen_dominantes[columna],
            ancho,
            label=etiqueta
        )

        ejes[0].bar_label(
            barras,
            padding=3,
            fontsize=9
        )

    ejes[0].set_xticks(
        posiciones,
        orden_experimentos
    )

    ejes[0].set_yscale("log")
    ejes[0].set_ylabel(
        "Número de reglas — escala logarítmica"
    )
    ejes[0].set_title(
        "Reglas descubiertas y reproducibles"
    )
    ejes[0].legend()
    ejes[0].grid(
        axis="y",
        alpha=0.25
    )

    ejes[1].plot(
        orden_experimentos,
        (
            tabla_resumen_dominantes[
                "proporcion_reproducibles"
            ]
            * 100
        ),
        marker="o",
        linewidth=2,
        label="Reproducibles / finales"
    )

    ejes[1].plot(
        orden_experimentos,
        (
            tabla_resumen_dominantes[
                "proporcion_estabilidad_alta"
            ]
            * 100
        ),
        marker="s",
        linewidth=2,
        label="Estabilidad alta / finales"
    )

    for indice, fila in (
        tabla_resumen_dominantes.iterrows()
    ):
        ejes[1].annotate(
            (
                f"{fila['proporcion_reproducibles'] * 100:.2f}%"
            ),
            (
                indice,
                fila[
                    "proporcion_reproducibles"
                ] * 100
            ),
            xytext=(0, 7),
            textcoords="offset points",
            ha="center"
        )

        ejes[1].annotate(
            (
                f"{fila['proporcion_estabilidad_alta'] * 100:.2f}%"
            ),
            (
                indice,
                fila[
                    "proporcion_estabilidad_alta"
                ] * 100
            ),
            xytext=(0, -14),
            textcoords="offset points",
            ha="center"
        )

    ejes[1].set_ylabel("Porcentaje")
    ejes[1].set_title(
        "Proporción de reglas reproducibles"
    )
    ejes[1].legend()
    ejes[1].grid(alpha=0.25)

    fig.tight_layout()

    fig.savefig(
        f"{carpeta_graficas_dominantes}/"
        "01_cantidad_y_reproducibilidad.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    # ============================================================
    # GRÁFICA 2. DISTRIBUCIÓN DE CALIDAD
    # ============================================================

    metricas_grafica = [
        ("fitness", "Fitness"),
        ("confianza", "Confianza"),
        ("lift", "Lift"),
        ("factor_certeza", "Factor de certeza")
    ]

    fig, ejes = plt.subplots(
        2,
        2,
        figsize=(13, 9)
    )

    for eje, (
        metrica,
        etiqueta
    ) in zip(
        ejes.flatten(),
        metricas_grafica
    ):
        datos = [
            catalogos_dominantes[
                experimento
            ]["reproducible"][
                metrica
            ].dropna().to_numpy()
            for experimento in orden_experimentos
        ]

        eje.boxplot(
            datos,
            labels=orden_experimentos,
            showmeans=True,
            meanline=True
        )

        eje.set_title(etiqueta)
        eje.set_ylabel(etiqueta)
        eje.grid(
            axis="y",
            alpha=0.25
        )

    fig.suptitle(
        "Calidad de las reglas dominantes reproducibles"
    )

    fig.tight_layout()

    fig.savefig(
        f"{carpeta_graficas_dominantes}/"
        "02_distribucion_calidad.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    # ============================================================
    # GRÁFICA 3. SOPORTE Y CONFIANZA
    # ============================================================

    colores_experimentos = {
        "50 partículas": "steelblue",
        "100 partículas": "darkorange",
        "500 partículas": "seagreen"
    }

    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    for experimento in orden_experimentos:
        catalogo = catalogos_dominantes[
            experimento
        ]["reproducible"]

        ax.scatter(
            catalogo["soporte_XY"],
            catalogo["confianza"],
            alpha=0.55,
            s=35,
            color=colores_experimentos[
                experimento
            ],
            label=experimento
        )

    ax.axhline(
        0.60,
        linestyle="--",
        color="gray",
        alpha=0.6,
        label="Confianza mínima"
    )

    ax.axvline(
        0.001,
        linestyle=":",
        color="darkred",
        alpha=0.6,
        label="Soporte 0.001"
    )

    ax.set_xlabel(
        "Soporte de la regla dominante"
    )

    ax.set_ylabel(
        "Confianza"
    )

    ax.set_title(
        "Relación entre soporte y confianza"
    )

    ax.grid(alpha=0.25)
    ax.legend()

    fig.tight_layout()

    fig.savefig(
        f"{carpeta_graficas_dominantes}/"
        "03_soporte_confianza.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    # ============================================================
    # GRÁFICA 4. LONGITUD DE LAS REGLAS ESTABLES
    # ============================================================

    tabla_proporciones_longitud = (
        tabla_longitud_dominantes
        .pivot(
            index="experimento",
            columns="numero_condiciones",
            values="proporcion"
        )
        .reindex(orden_experimentos)
        .fillna(0)
    )

    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    base = np.zeros(
        len(tabla_proporciones_longitud)
    )

    for longitud in tabla_proporciones_longitud.columns:
        valores = (
            tabla_proporciones_longitud[
                longitud
            ].to_numpy()
        )

        barras = ax.bar(
            tabla_proporciones_longitud.index,
            valores,
            bottom=base,
            label=f"|X| = {longitud}"
        )

        for barra, valor, inicio in zip(
            barras,
            valores,
            base
        ):
            if valor >= 0.05:
                ax.text(
                    barra.get_x()
                    + barra.get_width() / 2,
                    inicio + valor / 2,
                    f"{valor * 100:.1f}%",
                    ha="center",
                    va="center",
                    fontsize=9
                )

        base += valores

    ax.set_ylabel(
        "Proporción de reglas de estabilidad alta"
    )

    ax.set_title(
        "Complejidad de las reglas dominantes estables"
    )

    ax.set_ylim(0, 1)
    ax.legend(
        title="Número de condiciones"
    )
    ax.grid(
        axis="y",
        alpha=0.25
    )

    fig.tight_layout()

    fig.savefig(
        f"{carpeta_graficas_dominantes}/"
        "04_longitud_reglas.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    # 6. Guardar resultados consolidados
    resultados_graficas_dominantes = {
        "resumen_general": (
            tabla_resumen_dominantes
        ),
        "resumen_longitud": (
            tabla_longitud_dominantes
        ),
        "resumen_metodo_consecuente": (
            resumen_metodo_consecuente
        )
    }

    joblib.dump(
        resultados_graficas_dominantes,
        (
            f"{carpeta_graficas_dominantes}/"
            "resultados_graficas_dominantes.pkl"
        )
    )

    print("=" * 80)
    print("RESUMEN DE REGLAS DOMINANTES")
    print("=" * 80)

    display(
        tabla_resumen_dominantes.round({
            "proporcion_reproducibles": 4,
            "proporcion_estabilidad_alta": 4,
            "reglas_por_particula": 2,
            "fitness_mediano_reproducible": 6,
            "confianza_mediana_reproducible": 6,
            "lift_mediano_reproducible": 6
        })
    )

    print("\nResumen por consecuente y método:")

    display(
        resumen_metodo_consecuente.round(6)
    )

    print(
        f"\nGráficas guardadas en:\n"
        f"{carpeta_graficas_dominantes}"
    )
else:
    print("Graficas historicas de reglas dominantes omitidas.")
